In [ ]:
# ============================================================
# Cell 0 — Install + Setup
# ============================================================
!pip install -q together gdown pillow spacy
!python -m spacy download en_core_web_sm -q

import os, json, base64, re, time, random, math
from io import BytesIO
from collections import defaultdict, Counter
from pathlib import Path
from PIL import Image
import torch
from together import Together
import spacy

# ── Settings ─────────────────────────────────────────────────
TOGETHER_API_KEY = ""           # <-- paste your Together.ai key

DRIVE_IMAGES_DIR = "/content/drive/MyDrive/RelCheck_Data/images"
RBENCH_PATH      = "/content/drive/MyDrive/RelCheck_Data/rbench_data.json"

N_IMAGES = 20           # images to load for testing (20 is fast)
RANDOM_SEED = 42

VLM_MODEL    = "Qwen/Qwen3-VL-8B-Instruct"
MAVERICK_VLM = VLM_MODEL       # same model for now
LLM_MODEL    = "meta-llama/Llama-3.3-70B-Instruct-Turbo"

# Dummy path so pipeline functions don't crash on ENRICHED_PATH ref
SAVE_DIR     = "/content/detect_test_outputs"
os.makedirs(SAVE_DIR, exist_ok=True)

# ── Init ─────────────────────────────────────────────────────
os.environ["TOGETHER_API_KEY"] = TOGETHER_API_KEY
client  = Together(api_key=TOGETHER_API_KEY)
DEVICE  = "cuda" if torch.cuda.is_available() else "cpu"
random.seed(RANDOM_SEED)
nlp     = spacy.load("en_core_web_sm")
print(f"Device: {DEVICE}")

# ── LLM / VLM call helpers ───────────────────────────────────
def llm_call(messages, model=LLM_MODEL, max_tokens=600, retries=3):
    """Text-only LLM call via Together.ai."""
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=model, messages=messages,
                temperature=0.0, max_tokens=max_tokens,
            )
            return resp.choices[0].message.content
        except Exception as e:
            if attempt == retries - 1:
                print(f"  [LLM] failed: {e}")
                return None
            time.sleep(2 ** attempt)
    return None

def vlm_call(messages, max_tokens=800, retries=3, model=None):
    """VLM call (image+text) via Together.ai."""
    _model = model or VLM_MODEL
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=_model, messages=messages,
                temperature=0.0, max_tokens=max_tokens,
            )
            return resp.choices[0].message.content
        except Exception as e:
            if attempt == retries - 1:
                print(f"  [VLM] failed: {e}")
                return None
            time.sleep(2 ** attempt)
    return None

# Stubs for GPU models not loaded in this notebook
vitpose_model     = None
vitpose_processor = None
gdino_model       = None
gdino_processor   = None

print("Setup complete.")


In [ ]:
# ============================================================
# Cell 1 — Mount Drive + Load Images + R-Bench Data
# ============================================================
from google.colab import drive
drive.mount("/content/drive")

# ============================================================
# Cell 2 -- Load Images + R-Bench Data
# ============================================================
# Auto-downloads R-Bench if not present on Drive.

import pathlib

# ── Download + build rbench_data.json if missing ──
if not os.path.exists(RBENCH_PATH):
    print("R-Bench data not found on Drive. Downloading...")

    RBENCH_DIR = "/content/R-Bench"
    if not os.path.exists(RBENCH_DIR):
        get_ipython().system(f"git clone https://github.com/mrwu-mac/R-Bench {RBENCH_DIR}")
        print("  Cloned R-Bench repo")

    # Download annotations zip from Google Drive
    ANNOTATIONS_ZIP = f"{RBENCH_DIR}/rbench_annotations.zip"
    ANNOTATIONS_DIR = f"{RBENCH_DIR}/data"
    if not os.path.exists(ANNOTATIONS_DIR) or len(list(pathlib.Path(ANNOTATIONS_DIR).glob("*.json"))) == 0:
        DRIVE_FILE_ID = "1sqO0MWBg_HXp5cIKb-nstjNEEk5crUWH"
        get_ipython().system(f'gdown --id {DRIVE_FILE_ID} -O {ANNOTATIONS_ZIP} -q')
        import zipfile
        with zipfile.ZipFile(ANNOTATIONS_ZIP, 'r') as z:
            z.extractall(RBENCH_DIR)
        print("  Downloaded and extracted annotations")

    # Find image-level annotations
    all_jsons = sorted(pathlib.Path(RBENCH_DIR).rglob("*.json"))
    rbench_raw = None
    for f in all_jsons:
        fname = f.name.lower()
        if "image-level" in fname or "image_level" in fname:
            try:
                with open(f) as fh:
                    data = json.load(fh)
                if isinstance(data, list) and len(data) > 50:
                    rbench_raw = data
                    print(f"  Found annotations: {f.name} ({len(data)} entries)")
                    break
            except:
                pass

    # Fallback: any large JSON list with question-like keys
    if rbench_raw is None:
        for f in all_jsons:
            try:
                with open(f) as fh:
                    data = json.load(fh)
                if isinstance(data, list) and len(data) > 100:
                    sample = str(data[0]).lower()
                    if any(k in sample for k in ['question', 'text', 'label', 'answer']):
                        rbench_raw = data
                        print(f"  Found annotations: {f.name} ({len(data)} entries)")
                        break
            except:
                pass

    if rbench_raw is None:
        raise FileNotFoundError(
            "Could not find R-Bench annotations. Try manually downloading from:\n"
            "https://drive.google.com/file/d/1sqO0MWBg_HXp5cIKb-nstjNEEk5crUWH/view"
        )

    # Normalize entries
    def _get(entry, keys, default=None):
        for k in keys:
            if k in entry:
                return entry[k]
        return default

    normalized = []
    for entry in rbench_raw:
        image_filename = _get(entry, ["image", "img", "image_file"], "")
        if image_filename:
            img_id = os.path.splitext(os.path.basename(image_filename))[0]
        else:
            img_id = str(_get(entry, ["image_id", "img_id", "id"], ""))

        question = str(_get(entry, ["text", "question", "q"], ""))
        answer = str(_get(entry, ["label", "answer", "gt"], "")).strip().lower()
        if answer in ("1", "true"):
            answer = "yes"
        elif answer in ("0", "false"):
            answer = "no"

        if question and answer in ("yes", "no"):
            normalized.append({
                "image": image_filename,
                "image_id": img_id,
                "question": question,
                "answer": answer,
            })

    # Save to Drive for future runs
    os.makedirs(os.path.dirname(RBENCH_PATH), exist_ok=True)
    with open(RBENCH_PATH, "w") as f:
        json.dump(normalized, f)
    print(f"  Saved {len(normalized)} normalized entries to {RBENCH_PATH}")

# ── Load R-Bench ──
with open(RBENCH_PATH) as f:
    rbench_data = json.load(f)
print(f"R-Bench: {len(rbench_data)} questions total")

rbench_by_image = defaultdict(list)
for entry in rbench_data:
    # Support both 'image' key (filename) and 'image_id' key
    img_key = entry.get('image', entry.get('image_id', ''))
    rbench_by_image[img_key].append(entry)
print(f"R-Bench unique images: {len(rbench_by_image)}")

# Find images on Drive that have R-Bench questions
cached_images = list(Path(DRIVE_IMAGES_DIR).glob("*.jpg")) + \
                list(Path(DRIVE_IMAGES_DIR).glob("*.jpeg"))
cached_stems = {p.stem: p for p in cached_images}

matched = {}
for rb_key in rbench_by_image:
    stem = re.sub(r'\.(jpg|jpeg|png)$', '', rb_key)
    if stem in cached_stems:
        matched[stem] = {
            "path": str(cached_stems[stem]),
            "questions": rbench_by_image[rb_key],
        }

print(f"Images on Drive with R-Bench questions: {len(matched)}")

if len(matched) == 0:
    print("\n  WARNING: No images found on Drive!")
    print(f"  Expected images in: {DRIVE_IMAGES_DIR}")
    print("  You need to download nocaps images first.")
    print("  The R-Bench images are from nocaps validation set.")

# Sample N_IMAGES (or use all)
all_ids = list(matched.keys())
if N_IMAGES and len(all_ids) > N_IMAGES:
    selected_ids = random.sample(all_ids, N_IMAGES)
    print(f"Sampled {N_IMAGES} of {len(all_ids)} available images")
else:
    selected_ids = all_ids
    print(f"Using all {len(selected_ids)} available images")

# Load images into memory + question mapping
images = {}
img_to_questions = {}
for img_id in selected_ids:
    info = matched[img_id]
    images[img_id] = Image.open(info["path"]).convert("RGB")
    img_to_questions[img_id] = info["questions"]

total_q = sum(len(qs) for qs in img_to_questions.values())
print(f"\nLoaded {len(images)} images")
print(f"R-Bench questions: {total_q} ({total_q/len(images):.1f} per image)")


In [ ]:
# ============================================================
# Cell 1b — Load GDino (for bbox visualization only)
# ============================================================
# Optional but recommended — shows which objects were detected in each image.
# Does NOT affect detection logic (pipeline still uses empty KB).

from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection

GDINO_ID = "IDEA-Research/grounding-dino-tiny"
print("Loading GroundingDINO-tiny for visualization...")
gdino_processor = AutoProcessor.from_pretrained(GDINO_ID)
gdino_model     = AutoModelForZeroShotObjectDetection.from_pretrained(GDINO_ID).to(DEVICE)
print(f"  GroundingDINO loaded on {DEVICE}")

def _run_gdino(pil_image, text_queries, threshold=0.25):
    """Run GDino on image with a list of entity queries. Returns list of (label, box_norm, score)."""
    if gdino_model is None or not text_queries:
        return []
    query_str = " . ".join(q.lower().strip(" .") for q in text_queries if q.strip()) + " ."
    try:
        inputs = gdino_processor(images=pil_image, text=query_str, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            outputs = gdino_model(**inputs)
        W, H = pil_image.size
        results = gdino_processor.post_process_grounded_object_detection(
            outputs,
            inputs["input_ids"],
            box_threshold=threshold,
            text_threshold=threshold,
            target_sizes=[(H, W)],
        )[0]
        detections = []
        for box, score, label in zip(results["boxes"], results["scores"], results["labels"]):
            x1, y1, x2, y2 = box.tolist()
            detections.append({
                "label": label,
                "score": float(score),
                "box":   [x1/W, y1/H, x2/W, y2/H],   # normalised
                "box_px": [x1, y1, x2, y2],
            })
        return detections
    except Exception as e:
        print(f"  GDino error: {e}")
        return []


In [ ]:
# ============================================================
# Cell 2 — Pipeline Functions
# ============================================================
# Extracted from RelCheck_600 Cell 7.
# Includes: triple extraction, VQA verification, correction.
# Does NOT include enrichment (BLIP-2 path) or enrich_caption_v3.
# GDino/ViTPose are not loaded — empty KB triggers full-image VQA fallback.
# ============================================================

ENRICHED_PATH = f"{SAVE_DIR}/enriched.json"   # placeholder to avoid NameError
SHORT_CAPTION_THRESHOLD = 30

TRIPLE_EXTRACT_PROMPT = """Extract all relational claims from this image caption as structured triples.

Caption: "{caption}"

For each claim about how two entities relate, output a JSON object with:
  "subject"  : the entity performing or described first
  "relation" : the relationship word or short phrase (keep it brief, 1-4 words)
  "object"   : the entity being related to
  "type"     : one of SPATIAL | ACTION | ATTRIBUTE
    SPATIAL   = positional/directional (left of, right of, above, below, on, under,
                in front of, behind, next to, inside, near)
    ACTION    = dynamic interaction (riding, holding, eating, carrying, walking beside,
                playing with, pushing, throwing, sitting on, standing on)
    ATTRIBUTE = descriptive state (wearing, covered in, holding [as possession],
                decorated with, attached to, painted on)

Rules:
- Only include claims that describe a relationship BETWEEN two distinct entities.
- Skip generic single-entity descriptions ("a dog is brown").
- Use the shortest natural phrasing for relation (e.g. "on" not "is sitting on top of").
- subject and object should be short noun phrases (1-3 words).

Output ONLY a valid JSON array. No explanation. No markdown.
Example: [{{"subject": "dog", "relation": "left of", "object": "cat", "type": "SPATIAL"}},
          {{"subject": "man", "relation": "riding", "object": "horse", "type": "ACTION"}}]

If no relational claims exist, output: []"""


# Used by _apply_triple_correction — Llama edits exactly one relation in the caption.
# Intentionally does NOT require the exact wrong_phrase to appear verbatim, because
# the triple extractor may normalise the verb (e.g. extracts "riding" but caption
# says "astride"). Llama finds the contextually correct phrase and replaces it.
TRIPLE_CORRECT_PROMPT = """Edit exactly one relationship word or phrase in this caption.

Caption: "{caption}"

Task: The caption incorrectly describes how {subj} and {obj} relate to each other.
Find the word or phrase that expresses this relationship between {subj} and {obj}.
It may not be exactly "{wrong_phrase}" — the caption might use a synonym or different phrasing.
Replace it with: "{correct_phrase}"

Rules:
- Change ONLY the relationship word/phrase between {subj} and {obj}.
- Do NOT change any other words, word order, punctuation, or sentence structure.
- Do NOT add, remove, or reorder any sentences.
- The corrected caption must be the same length (±10%) as the original.
- Output the FULL corrected caption only. No explanation, no prefix, no quotes."""




# Batched correction prompt — Llama fixes ALL confirmed hallucinations in one call.
# Advantages over per-triple surgical edit:
#   1. Handles full sentence removal (e.g. "The treadmill is located in the center")
#   2. No dangling text artifacts from character-level replacement
#   3. Context-aware rewrite — can restructure sentences around false claims
BATCH_CORRECT_PROMPT = """You are correcting an image caption that contains confirmed factual errors.

Original caption:
"{caption}"

CONFIRMED ERRORS (fix ALL of these — do not skip any):
{error_list}

Rules:
1. Fix EVERY error listed above — address each one explicitly.
2. If an entity does not exist in the image, remove the ENTIRE sentence(s) about it.
3. If a relationship is wrong, correct only that relationship — keep surrounding text intact.
4. Do NOT add new information not present in the original caption.
5. Do NOT shorten the caption by removing correct information.
6. The output must be fluent, grammatical English — no dangling phrases.
7. CONSISTENCY: If you change a word (e.g. "pulling" → "pushing"), change ALL occurrences
   of that word throughout the caption, not just the first. A caption that says both
   "pushing" and "pulling" for the same subject is worse than the original.
8. Output the FULL corrected caption ONLY. No explanation, no prefix, no quotes."""

# -- Utility functions -------------------------------------------------


def vlm_call(messages, max_tokens=800, retries=3, model=None):
    """Call a VLM on Together.ai. model defaults to VLM_MODEL (Qwen3-VL-8B).
    Pass model=MAVERICK_VLM for the higher-quality contrastive VQA call."""
    _model = model or VLM_MODEL
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=_model,
                messages=messages,
                temperature=0.0,
                max_tokens=max_tokens,
            )
            return resp.choices[0].message.content.strip()
        except Exception as e:
            if attempt < retries - 1:
                wait = 2 ** attempt
                print(f"  VLM API error (attempt {attempt+1}): {e} — retrying in {wait}s")
                time.sleep(wait)
            else:
                print(f"  VLM API failed after {retries} attempts: {e}")
                return None

def levenshtein(s1, s2):
    """Character-level edit distance for computing edit rate."""
    if s1 == s2: return 0
    m, n = len(s1), len(s2)
    dp = list(range(n + 1))
    for i in range(1, m + 1):
        prev = dp[0]
        dp[0] = i
        for j in range(1, n + 1):
            temp = dp[j]
            dp[j] = prev if s1[i-1] == s2[j-1] else 1 + min(prev, dp[j], dp[j-1])
            prev = temp
    return dp[n]


def _check_entity_exists_vqa(entity, pil_image, retries=2):
    """
    Full-image VQA: ask Maverick whether an entity actually exists in the image.
    Used as fallback when GDino can't detect the entity (and therefore crop VQA
    is unavailable). Returns True (exists), False (does not exist), None (uncertain).
    """
    if pil_image is None:
        return None
    buf = BytesIO()
    pil_image.convert("RGB").save(buf, format="JPEG", quality=85)
    b64 = base64.b64encode(buf.getvalue()).decode()

    questions = [
        f"Is there a {entity} visible anywhere in this image? Answer only YES or NO.",
        f"Can you see a {entity} in this scene? Look carefully. Answer YES or NO only.",
    ]
    yes_v = no_v = 0
    for q in questions:
        r = vlm_call([{"role": "user", "content": [
            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}"}},
            {"type": "text", "text": q},
        ]}], max_tokens=5)
        if not r:
            continue
        rl = r.strip().lower()
        if "yes" in rl and "no" not in rl:
            yes_v += 1
        elif "no" in rl:
            no_v += 1
    total = yes_v + no_v
    if total == 0:
        return None
    if no_v > yes_v:
        return False   # majority says entity absent
    if yes_v > no_v:
        return True    # majority says entity present
    return None        # tie → uncertain


def _query_correct_spatial_relation(subj, obj, kb, pil_image):
    """
    Ask Maverick what the actual spatial relationship between subj and obj IS.
    Called when VQA has already confirmed the caption's claimed relation is wrong.
    Returns a short relation phrase (e.g. "to the right of") or None on failure.
    """
    if pil_image is None:
        return None
    subj_box = _find_best_bbox(subj, kb)
    obj_box  = _find_best_bbox(obj,  kb)

    using_crop = bool(subj_box and obj_box)
    if using_crop:
        crop = _crop_to_bboxes(pil_image, subj_box, obj_box, padding=0.25)
        buf = BytesIO()
        crop.convert("RGB").save(buf, format="JPEG", quality=85)
        img_b64 = base64.b64encode(buf.getvalue()).decode()
        region_hint = "this cropped region"
    else:
        # GDino missed one or both entities (common for food, tableware, small objects).
        # Use the full image — less precise but still usable for VQA.
        buf = BytesIO()
        pil_image.convert("RGB").save(buf, format="JPEG", quality=85)
        img_b64 = base64.b64encode(buf.getvalue()).decode()
        region_hint = "the full image"

    prompt = (
        f"Look at this image. Where is the {obj} relative to the {subj}? "
        f"Reply with ONLY a short spatial phrase "
        f"(e.g. 'to the left of', 'to the right of', 'above', 'below', "
        f"'in front of', 'behind', 'next to'). "
        f"If you cannot determine the relationship clearly, reply 'unknown'."
    )
    raw = vlm_call([{"role": "user", "content": [
        {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{img_b64}"}},
        {"type": "text", "text": prompt},
    ]}], max_tokens=15)

    if raw:
        r = raw.strip().strip('"').lower()
        if r != "unknown" and len(r.split()) <= 5:
            return r
    return None


def normalize_entity(text):
    """Lowercase, strip articles for fuzzy matching."""
    if not text:
        return ""
    text = text.lower().strip()
    for art in ["a ", "an ", "the ", "some "]:
        if text.startswith(art):
            text = text[len(art):]
    return text.strip()


# -- Deterministic spatial contradiction detection ----------------------

# Opposite spatial relations for direct contradiction detection
SPATIAL_OPPOSITES = {
    "left":         "right",
    "right":        "left",
    "above":        "below",
    "below":        "above",
    "on top of":    "below",
    "under":        "above",
    "over":         "under",
    "in front of":  "behind",
    "behind":       "in front of",
    "to the left":  "to the right",
    "to the right": "to the left",
}

# Spatial relation synonym groups — relations within the same group are
# treated as equivalent (surface-contact semantics).  This prevents the
# verifier from flagging "on a table" when geometry says "above table".
_SPATIAL_SYNONYM_GROUPS = [
    {"on", "above", "on top of", "over"},      # surface contact / vertical above
    {"under", "below", "beneath", "underneath"}, # vertical below
    {"left", "to the left", "to the left of"},
    {"right", "to the right", "to the right of"},
    {"in front of", "before"},
    {"behind", "in back of"},
]

def _spatial_synonyms(rel):
    """Return the synonym set that contains *rel*, or {rel} if none."""
    rel_lower = rel.lower().strip()
    for group in _SPATIAL_SYNONYM_GROUPS:
        if rel_lower in group:
            return group
    return {rel_lower}

# Regex to extract spatial triples from free text
# Matches: '<subj> is [to the] left/right/above/below [of] <obj>'
_SPATIAL_TRIPLE_RE = re.compile(
    r"([a-z][a-z\s]{1,25}?)\s+(?:is\s+)?(?:to\s+the\s+)?"
    r"(left|right|above|below|on top of|under|over|in front of|behind)"
    r"(?:\s+of)?\s+([a-z][a-z\s]{1,25})",
    re.IGNORECASE,
)


# ── Action Geometry Taxonomy (Tier 1) ──────────────────────────────────
# Physical families whose prerequisites can be checked from bounding boxes
# alone (no pose estimation needed). If the geometric rule FAILS, the action
# is a confirmed hallucination. If it PASSES, we still run VQA (necessary
# condition, not sufficient).

ACTION_GEOMETRY_TAXONOMY = {
    # ── Tier 1: object-level bboxes only ──
    "mounting": {
        "verbs": {"riding", "sitting on", "standing on", "straddling",
                  "mounted on", "perched on", "atop", "on top of",
                  "perching on", "seated on", "crouching on"},
        "geometric_rule": "subject_above_object",
        "needs_keypoints": False,
    },
    "containment": {
        "verbs": {"inside", "in", "enclosed by", "covered by",
                  "contained in", "within", "trapped in", "wrapped in"},
        "geometric_rule": "subject_inside_object",
        "needs_keypoints": False,
    },
    "adjacency": {
        "verbs": {"next to", "beside", "near", "alongside", "adjacent to",
                  "close to", "leaning on", "leaning against"},
        "geometric_rule": "bboxes_close",
        "needs_keypoints": False,
    },
    # ── Tier 2: requires ViTPose keypoints ──
    "grasping": {
        "verbs": {"holding", "carrying", "picking up", "pulling", "pushing",
                  "grabbing", "gripping", "lifting", "dragging", "clutching",
                  "catching", "throwing", "tossing"},
        "geometric_rule": "wrist_near_object",
        "needs_keypoints": True,
    },
    "consuming": {
        "verbs": {"eating", "drinking", "tasting", "licking", "biting",
                  "sipping", "chewing", "feeding on"},
        "geometric_rule": "nose_near_object",
        "needs_keypoints": True,
    },
}

# Build a reverse lookup: verb → family name
_VERB_TO_FAMILY = {}
for _fam, _spec in ACTION_GEOMETRY_TAXONOMY.items():
    for _v in _spec["verbs"]:
        _VERB_TO_FAMILY[_v] = _fam


def _classify_action_family(relation_verb):
    """Map a relation verb to its physical family (or None if no rule exists)."""
    rel = relation_verb.strip().lower()
    if rel in _VERB_TO_FAMILY:
        return _VERB_TO_FAMILY[rel]
    # Fuzzy: only match multi-word verbs where the FULL known verb appears
    # as a substring in the relation. Single-word verbs like "in" are too
    # ambiguous for fuzzy matching ("pushing in" ≠ containment).
    for verb, fam in _VERB_TO_FAMILY.items():
        if len(verb.split()) >= 2 and verb in rel:
            return fam
    return None


# ── COCO keypoint indices ──
KP_NOSE        = 0
KP_LEFT_WRIST  = 9
KP_RIGHT_WRIST = 10


def _get_person_keypoints(pil_image, person_box_norm):
    """
    Run ViTPose on a detected person to get 17 COCO keypoints.

    Args:
        pil_image: PIL Image
        person_box_norm: [x1, y1, x2, y2] normalised to [0, 1]

    Returns:
        dict with 'keypoints' (17×2 pixel coords) and 'scores' (17,)
        or None if detection fails.
    """
    if vitpose_model is None:
        return None   # ViTPose not loaded — skip keypoint verification
    W, H = pil_image.size
    x1, y1, x2, y2 = person_box_norm
    # Convert to COCO format: [left, top, width, height] in pixels
    coco_box = [x1 * W, y1 * H, (x2 - x1) * W, (y2 - y1) * H]

    try:
        inputs = vitpose_processor(
            pil_image, boxes=[[coco_box]], return_tensors="pt"
        ).to(vitpose_model.device)

        with torch.no_grad():
            outputs = vitpose_model(**inputs)

        results = vitpose_processor.post_process_pose_estimation(
            outputs, boxes=[[coco_box]]
        )

        if results and results[0]:
            kp = results[0][0]
            keypoints = kp["keypoints"].cpu().numpy()   # (17, 2) in pixels
            scores    = kp["scores"].cpu().numpy()       # (17,)
            # Normalise to [0, 1]
            keypoints[:, 0] /= W
            keypoints[:, 1] /= H
            return {"keypoints": keypoints, "scores": scores}
    except Exception as e:
        print(f"    ViTPose error: {e}")
    return None


def _check_action_geometry(family, subj_box, obj_box, keypoints=None):
    """
    Test the geometric prerequisite for an action family.

    Args:
        family: one of "mounting", "containment", "adjacency",
                "grasping", "consuming"
        subj_box, obj_box: [x1, y1, x2, y2] normalised to [0, 1]
        keypoints: optional dict from _get_person_keypoints
                   (required for Tier 2 families)

    Returns:
        True  → prerequisite MET (inconclusive, still need VQA)
        False → prerequisite VIOLATED (hallucination confirmed)
        None  → cannot check (missing keypoints for Tier 2)
    """
    sx1, sy1, sx2, sy2 = subj_box  # subject
    ox1, oy1, ox2, oy2 = obj_box   # object

    s_cx, s_cy = (sx1 + sx2) / 2, (sy1 + sy2) / 2
    o_cx, o_cy = (ox1 + ox2) / 2, (oy1 + oy2) / 2
    s_h = sy2 - sy1
    o_h = oy2 - oy1
    o_w = ox2 - ox1

    # ── Tier 1: bbox-only checks ──

    if family == "mounting":
        # Subject should be above/on the object:
        #   1. Subject's bottom edge (sy2) should be near object's top region
        #      (within top 40% of object — generous to handle perspective)
        #   2. Horizontal overlap: subject and object x-ranges must intersect
        top_region = oy1 + 0.65 * o_h  # generous: rider's feet reach ~60% of horse height
        subject_bottom_in_top = sy2 <= top_region + 0.05  # small tolerance
        x_overlap = min(sx2, ox2) - max(sx1, ox1)
        has_x_overlap = x_overlap > 0.02 * max(o_w, 0.01)
        return subject_bottom_in_top and has_x_overlap

    elif family == "containment":
        # Subject bbox should be mostly inside object bbox (>50% overlap)
        inter_x = max(0, min(sx2, ox2) - max(sx1, ox1))
        inter_y = max(0, min(sy2, oy2) - max(sy1, oy1))
        inter_area = inter_x * inter_y
        subj_area = max((sx2 - sx1) * (sy2 - sy1), 1e-6)
        containment_ratio = inter_area / subj_area
        return containment_ratio > 0.50

    elif family == "adjacency":
        # Bboxes should be close — gap between nearest edges < 30% of avg size
        gap_x = max(0, max(sx1, ox1) - min(sx2, ox2))
        gap_y = max(0, max(sy1, oy1) - min(sy2, oy2))
        gap = (gap_x**2 + gap_y**2) ** 0.5
        avg_size = ((s_h + o_h) / 2 + ((sx2-sx1) + o_w) / 2) / 2
        return gap < 0.30 * max(avg_size, 0.01)

    # ── Tier 2: keypoint-based checks ──

    if keypoints is None:
        return None  # can't check without keypoints

    kp_xy = keypoints["keypoints"]   # (17, 2) normalised
    kp_sc = keypoints["scores"]      # (17,)
    KP_CONF_THRESH = 0.3             # minimum keypoint confidence

    if family == "grasping":
        # At least one wrist must be near/inside the object bbox.
        # Margin scales with object size (50% of object dimension on each side)
        # to handle perspective while avoiding matching distant wrists.
        margin_x = max(0.5 * o_w, 0.03)  # at least 3% of image
        margin_y = max(0.5 * o_h, 0.03)
        obj_expanded = [ox1 - margin_x, oy1 - margin_y,
                        ox2 + margin_x, oy2 + margin_y]
        for wrist_idx in [KP_LEFT_WRIST, KP_RIGHT_WRIST]:
            if kp_sc[wrist_idx] < KP_CONF_THRESH:
                continue
            wx, wy = kp_xy[wrist_idx]
            if (obj_expanded[0] <= wx <= obj_expanded[2] and
                obj_expanded[1] <= wy <= obj_expanded[3]):
                return True  # wrist near object — prerequisite met
        # Neither wrist is near the object
        return False

    elif family == "consuming":
        # Nose (proxy for mouth) must be near the object bbox.
        if kp_sc[KP_NOSE] < KP_CONF_THRESH:
            return None  # nose not detected — can't check
        nx, ny = kp_xy[KP_NOSE]
        # Margin scales with object size (75% — more generous because
        # mouth is below nose, and food can be held at varying angles)
        margin_x = max(0.75 * o_w, 0.04)
        margin_y = max(0.75 * o_h, 0.04)
        obj_expanded = [ox1 - margin_x, oy1 - margin_y,
                        ox2 + margin_x, oy2 + margin_y]
        if (obj_expanded[0] <= nx <= obj_expanded[2] and
            obj_expanded[1] <= ny <= obj_expanded[3]):
            return True  # nose near food/drink — prerequisite met
        return False

    return True  # unknown family — don't block


def _extract_spatial_triples_text(text):
    """
    Extract (subject, relation, object) spatial triples from free text.
    Returns list of (subj, rel, obj) tuples, all lowercase.
    """
    triples = []
    for m in _SPATIAL_TRIPLE_RE.finditer(text.lower()):
        subj = m.group(1).strip().rstrip(" ,;")
        rel  = m.group(2).strip()
        obj  = m.group(3).strip().rstrip(" ,;.")
        triples.append((subj, rel, obj))
    return triples


def _parse_spatial_facts(spatial_facts):
    """
    Parse the KB's spatial_facts list into (subj, rel, obj) tuples.
    spatial_facts lines look like: "'dog' is to the left of 'couch'"
    """
    parsed = []
    for fact in spatial_facts:
        fact_clean = fact.replace("'", "").replace('"', "")
        for m in _SPATIAL_TRIPLE_RE.finditer(fact_clean.lower()):
            subj = m.group(1).strip().rstrip(" ,;")
            rel  = m.group(2).strip()
            obj  = m.group(3).strip().rstrip(" ,;.")
            parsed.append((subj, rel, obj))
    return parsed


# Stop words and filler verbs stripped when extracting core noun from regex captures
_ENTITY_STOP = frozenset([
    "a","an","the","some","is","are","was","were","be","been","being",
    "sits","sit","standing","stand","positioned","position","placed","place",
    "seen","located","appears","appear","lying","lies","lay","resting","rest",
])

def _core_noun(text):
    """
    Extract the core noun from a regex-captured entity span.
    Strips leading articles, filler verbs, and trailing noise words.
    Returns first 1-2 meaningful words (the actual object/person name).
    e.g. 'a dog sits'           -> 'dog'
         'a dog is positioned'  -> 'dog'
         'the couch while a'    -> 'couch'
         'large red ball'       -> 'large red ball' (adjective + noun kept)
    """
    words = normalize_entity(text).split()
    # Drop leading stop words
    while words and words[0] in _ENTITY_STOP:
        words = words[1:]
    # Drop trailing stop words / noise
    while words and words[-1] in _ENTITY_STOP:
        words = words[:-1]
    # Cap at 3 words
    return " ".join(words[:3]).strip()


def _entity_matches(cap_entity, kb_entity):
    """
    Fuzzy match using core noun extraction.
    'a dog sits' matches 'dog', 'a dog is positioned', 'large dog', etc.
    """
    core_cap = _core_noun(cap_entity)
    core_kb  = _core_noun(kb_entity)
    if not core_cap or not core_kb:
        return False
    # Match if either core noun contains the other
    return (core_kb in core_cap) or (core_cap in core_kb)


def _check_spatial_contradictions(caption, spatial_facts):
    """
    Deterministically detect spatial contradictions between the caption
    and GroundingDINO bbox geometry.

    A contradiction exists when the caption states A <rel> B but the KB
    geometry says A <opposite_of_rel> B.  Uses fuzzy entity matching so
    'a dog sits' correctly matches KB entity 'dog'.

    Returns list of contradiction strings, e.g.:
      ["Caption says 'a dog sits left of couch' but geometry shows: 'dog right of couch'"]
    """
    if not spatial_facts:
        return []

    kb_triples = _parse_spatial_facts(spatial_facts)
    if not kb_triples:
        return []

    caption_triples = _extract_spatial_triples_text(caption)
    if not caption_triples:
        return []

    contradictions = []
    for (cap_subj, cap_rel, cap_obj) in caption_triples:
        cap_opposite = SPATIAL_OPPOSITES.get(cap_rel.lower())
        if not cap_opposite:
            continue

        for (kb_subj, kb_rel, kb_obj) in kb_triples:
            # Fuzzy entity match: 'a dog sits' matches 'dog'
            if not (_entity_matches(cap_subj, kb_subj) and _entity_matches(cap_obj, kb_obj)):
                continue
            # Contradiction: caption claims rel, KB says the opposite
            if kb_rel == cap_opposite:
                kb_fact_str = next(
                    (f for f in spatial_facts
                     if normalize_entity(kb_subj) in f.lower()
                     and normalize_entity(kb_obj) in f.lower()),
                    f"'{kb_subj}' {kb_rel} '{kb_obj}'"
                )
                contradictions.append(
                    f"Caption says '{cap_subj} {cap_rel} {cap_obj}' "
                    f"but geometry shows: {kb_fact_str}"
                )
                break  # one contradiction report per caption triple

    return contradictions
def _enrich_short_caption(img_id, caption, kb):
    """Full KB-guided enrichment for captions under 30 words."""
    hard    = "\n".join(f"- {f}" for f in kb["hard_facts"])    or "- None detected"
    spatial = "\n".join(f"- {f}" for f in kb["spatial_facts"]) or "- No spatial facts"
    visual  = kb["visual_description"][:800]                     or "- No visual description"

    prompt = ANALYSIS_PROMPT.format(
        caption=caption, hard_facts=hard,
        spatial_facts=spatial, visual_description=visual
    )
    improved = caption
    errors, missing = [], []

    raw = llm_call([{"role": "user", "content": prompt}], max_tokens=1000)
    if raw:
        raw = re.sub(r'^```json\s*', '', raw)
        raw = re.sub(r'```\s*$', '', raw)
        if raw.count('{') > raw.count('}'):
            raw += '"}' 
        try:
            result  = json.loads(raw)
            errors  = result.get("errors", [])
            missing = result.get("missing", [])
            cand    = result.get("improved_caption", "").strip().strip('"').strip("'")
            if cand and len(cand) >= 15:
                improved = cand
        except Exception:
            pass

    # Sentence cap
    if improved != caption:
        n_sent = len([s for s in re.split(r'[.!?]+', improved) if s.strip()])
        if n_sent > 10:  # prompt asks for 5-8 sentences; allow 2 slack
            improved = caption

    # Verification gate
    if improved != caption:
        counts  = Counter(l for l, _, _ in kb.get("detections", []))
        obj_str = ", ".join(f"{c}x {l}" for l, c in counts.most_common(10))
        rel_str = kb["visual_description"][:500]
        verdict = llm_call(
            [{"role": "user", "content": VERIFY_PROMPT.format(
                rewritten=improved, objects=obj_str, relationships=rel_str
            )}],
            max_tokens=50,
        )
        if verdict and verdict.upper().startswith("FAIL"):
            improved = caption

    edit_rate = levenshtein(caption, improved) / max(len(caption), len(improved), 1)
    return {
        "caption": caption, "corrected": improved,
        "errors": errors,   "missing": missing,
        "edit_rate": edit_rate,
        "status": "modified" if improved != caption else "unchanged",
        "mode": "enrich",
    }
    rel_norm = rel.lower()
    # Also check phrase variants: "left" matches "to the left", "left of", etc.
    rel_variants = {
        "left":  ["left"],  "right": ["right"],
        "above": ["above","on top of"],  "below": ["below","beneath","under"],
        "behind":["behind","in back of"], "in front of":["in front of","front"],
    }
    keywords = rel_variants.get(rel_norm, [rel_norm])
    for kw in keywords:
        if kw not in cap_lower:
            continue
        idx = cap_lower.find(kw)
        context = cap_lower[max(0, idx - 80): idx + 80]
        # If the relation keyword appears and EITHER entity is nearby, assume covered
        core_s = _core_noun(subj)
        core_o = _core_noun(obj)
        if core_s in context or core_o in context:
            return True
        # Also check synonyms in the context window
        for syn in _ENTITY_SYNONYMS.get(core_s, []):
            if syn in context:
                return True
        for syn in _ENTITY_SYNONYMS.get(core_o, []):
            if syn in context:
                return True
    return False



def _apply_geometry_corrections(caption, geo_contradictions):
    """
    Surgical fallback: directly replace wrong spatial phrases in the original caption
    using deterministic bbox geometry. Used when Maverick's full rewrite is rejected
    by the length guard (too compressed).

    Parses each geometry contradiction string:
      "Caption says 'X rel Y' but geometry shows: ..."
    Finds 'rel' near X/Y entities in caption and swaps it for the opposite.

    Returns (fixed_caption, n_fixed).
    """
    if not geo_contradictions:
        return caption, 0

    fixed = caption
    n_fixed = 0

    for contra in geo_contradictions:
        # Parse the wrong caption claim: "Caption says 'dog left of couch' but ..."
        m_cap = re.search(r"Caption says '(.+?)'", contra, re.IGNORECASE)
        if not m_cap:
            continue
        cap_claim = m_cap.group(1).strip()

        # Extract the spatial relation from the claim
        m_rel = _SPATIAL_TRIPLE_RE.search(cap_claim.lower())
        if not m_rel:
            continue
        wrong_rel = m_rel.group(2).strip()
        correct_rel = SPATIAL_OPPOSITES.get(wrong_rel)
        if not correct_rel:
            continue

        # Find the exact claim phrase in the caption (case-insensitive)
        claim_lower = cap_claim.lower()
        fixed_lower = fixed.lower()
        idx = fixed_lower.find(claim_lower)
        if idx < 0:
            # Phrase not found verbatim — try replacing just the relation keyword near entities
            rel_idx = fixed_lower.find(wrong_rel)
            if rel_idx < 0:
                continue
            # Check entity proximity (within 60 chars of the relation keyword)
            context = fixed_lower[max(0, rel_idx - 60): rel_idx + 60]
            subj = m_rel.group(1).strip()
            obj_ = m_rel.group(3).strip()
            core_s = _core_noun(subj)
            core_o = _core_noun(obj_)
            if not (core_s in context or core_o in context):
                continue
            # Replace just the relation keyword at rel_idx
            fixed = fixed[:rel_idx] + correct_rel + fixed[rel_idx + len(wrong_rel):]
            n_fixed += 1
            continue

        # Replace the relation keyword within the matched phrase
        phrase = fixed[idx: idx + len(cap_claim)]
        corrected_phrase = re.sub(
            r'\b' + re.escape(wrong_rel) + r'\b',
            correct_rel, phrase, flags=re.IGNORECASE, count=1,
        )
        if corrected_phrase != phrase:
            fixed = fixed[:idx] + corrected_phrase + fixed[idx + len(cap_claim):]
            n_fixed += 1

    return fixed, n_fixed



def _caption_name_for(kb_entity, cap_lower):
    """
    Return the synonym of kb_entity that actually appears in cap_lower.
    Falls back to kb_entity's core noun if nothing matches.
    Used by the addendum builder to produce "the man is left of the car"
    instead of "the person is left of the car" when caption says "man".
    """
    core = _core_noun(kb_entity)
    if not core:
        return kb_entity
    if core in cap_lower:
        return core
    for syn in _ENTITY_SYNONYMS.get(core, []):
        if syn in cap_lower:
            return syn
    return core  # fallback: use KB label


# COCO / GroundingDINO class synonyms — maps detector labels to caption synonyms.
# Used by _relation_already_expressed and _build_spatial_addendum to match
# caption entity names (e.g. "man") against KB labels (e.g. "person").
_ENTITY_SYNONYMS = {
    "person":       ["person","man","woman","child","boy","girl","individual","human","people"],
    "car":          ["car","vehicle","automobile","sedan","suv","truck"],
    "couch":        ["couch","sofa","settee"],
    "chair":        ["chair","seat","stool"],
    "dog":          ["dog","puppy","canine"],
    "cat":          ["cat","kitten","feline"],
    "horse":        ["horse","pony"],
    "bicycle":      ["bicycle","bike","cycle"],
    "motorcycle":   ["motorcycle","motorbike"],
    "truck":        ["truck","lorry","van"],
    "bus":          ["bus","coach"],
    "boat":         ["boat","ship","vessel"],
    "airplane":     ["airplane","plane","aircraft","jet"],
    "bird":         ["bird","pigeon","seagull","sparrow"],
    "cow":          ["cow","cattle","bull"],
    "sheep":        ["sheep","lamb"],
    "bench":        ["bench","seat"],
    "dining table": ["table","dining table","desk"],
    "tv":           ["tv","television","monitor","screen","display"],
    "laptop":       ["laptop","computer","notebook"],
    "cell phone":   ["phone","cell phone","smartphone","mobile"],
    "backpack":     ["backpack","bag","rucksack"],
    "handbag":      ["handbag","bag","purse"],
    "sports ball":  ["ball"],
    "cup":          ["cup","mug","glass"],
    "bottle":       ["bottle"],
    "bowl":         ["bowl"],
    "book":         ["book","magazine"],
    "vase":         ["vase"],
    "clock":        ["clock","watch"],
    "potted plant": ["plant","flower","pot"],
    "fire hydrant": ["fire hydrant","hydrant"],
    "traffic light":["traffic light","stoplight"],
    "umbrella":     ["umbrella"],
    "frisbee":      ["frisbee"],
    "skateboard":   ["skateboard","board"],
    "surfboard":    ["surfboard","board"],
    "skis":         ["skis","ski"],
    "kite":         ["kite"],
    "baseball bat": ["bat","baseball bat"],
    "tennis racket":["racket","tennis racket"],
    "suitcase":     ["suitcase","luggage","bag"],
    "tie":          ["tie","necktie"],
    "keyboard":     ["keyboard"],
    "mouse":        ["mouse"],
    "remote":       ["remote","controller"],
    "pizza":        ["pizza"],
    "cake":         ["cake"],
    "sandwich":     ["sandwich"],
    "refrigerator": ["refrigerator","fridge"],
    "oven":         ["oven","stove"],
    "sink":         ["sink"],
    "toilet":       ["toilet"],
    "bed":          ["bed"],

    # ── Genuine semantic synonyms (same object, different words) ──
    # Only true equivalents — NOT GDino mislabeling hacks, which would
    # match wrong objects and produce incorrect geometry verification.
    "barbell":      ["barbell","dumbbell","weight","bar"],   # all refer to free weights
    "dumbbell":     ["dumbbell","barbell","weight"],
    "treadmill":    ["treadmill","exercise machine"],
    "sink":         ["sink","basin","washbasin","washbowl","double basin"],
    "basin":        ["basin","sink","washbasin"],
    "faucet":       ["faucet","tap","spigot"],
    "box":          ["box","crate","chest","container","drawer"],
    "crate":        ["crate","box","container"],
    "basket":       ["basket","bin","hamper"],
    "shelf":        ["shelf","rack","ledge"],
    "barrel":       ["barrel","drum","keg"],
    "cart":         ["cart","trolley","wagon"],
    "couch":        ["couch","sofa","settee","loveseat"],
}


def _relation_already_expressed(subj, rel, obj, cap_lower):
    """
    Heuristic: return True if the caption already states subj <rel> obj
    OR the equivalent reversed relation (obj <opposite> subj).
    Uses a loose keyword proximity check.
    """
    rel_norm = rel.lower()
    # Also check phrase variants: "left" matches "to the left", "left of", etc.
    rel_variants = {
        "left":  ["left"],  "right": ["right"],
        "above": ["above","on top of"],  "below": ["below","beneath","under"],
        "behind":["behind","in back of"], "in front of":["in front of","front"],
    }
    keywords = rel_variants.get(rel_norm, [rel_norm])
    for kw in keywords:
        if kw not in cap_lower:
            continue
        idx = cap_lower.find(kw)
        context = cap_lower[max(0, idx - 80): idx + 80]
        # If the relation keyword appears and EITHER entity is nearby, assume covered
        core_s = _core_noun(subj)
        core_o = _core_noun(obj)
        if core_s in context or core_o in context:
            return True
        # Also check synonyms in the context window
        for syn in _ENTITY_SYNONYMS.get(core_s, []):
            if syn in context:
                return True
        for syn in _ENTITY_SYNONYMS.get(core_o, []):
            if syn in context:
                return True
    return False


def _build_spatial_addendum(corrected_caption, kb, max_facts=5):
    """
    Append missing KB spatial facts to the (corrected) caption.

    v2 strategy (relaxed):
      - REMOVE the requirement that both entities already appear in the caption.
        GroundingDINO bbox facts are deterministic ground truth; it's always
        safe to state them even if the captioner used different words or omitted
        an object. R-POPE asks about entities by their canonical name, so adding
        "the person is to the left of the car" helps the LLM-judge even if
        LLaVA called them "the man" and "the vehicle".
      - KEEP the deduplication check: if the relation is already expressed
        (with synonym-aware entity matching), skip to avoid repetition.

    Returns (new_caption_str, n_added) tuple.
    """
    spatial_facts = kb.get("spatial_facts", [])
    if not spatial_facts:
        return corrected_caption, 0

    cap_lower = corrected_caption.lower()
    missing = []

    for fact in spatial_facts:
        triples = _parse_spatial_facts([fact])
        if not triples:
            continue
        subj, rel, obj = triples[0]
        if not subj or not obj:
            continue

        # Skip if this spatial relation is already expressed in caption
        if _relation_already_expressed(subj, rel, obj, cap_lower):
            continue

        cap_subj = _caption_name_for(subj, cap_lower)
        cap_obj  = _caption_name_for(obj,  cap_lower)
        missing.append((subj, rel, obj, fact, cap_subj, cap_obj))
        if len(missing) >= max_facts:
            break

    if not missing:
        return corrected_caption, 0

    # Build addendum sentence
    fact_phrases = [f"the {cs} is {r} the {co}" for s, r, o, _, cs, co in missing]
    if len(fact_phrases) == 1:
        addendum = f"Spatially, {fact_phrases[0]}."
    elif len(fact_phrases) == 2:
        addendum = f"Spatially, {fact_phrases[0]}, and {fact_phrases[1]}."
    else:
        joined = ", ".join(fact_phrases[:-1]) + f", and {fact_phrases[-1]}"
        addendum = f"Spatially, {joined}."

    new_cap = corrected_caption.rstrip() + " " + addendum
    return new_cap, len(missing)
def _extract_triples(caption):
    """Use Llama to extract (subject, relation, object, type) triples.

    type field is inferred from the relation verb if Llama omits it or uses
    wrong capitalisation — prevents silent empty returns when Llama output
    is valid but slightly off-format.
    """
    _SPATIAL_WORDS = {"left","right","above","below","behind","front",
                      "beside","next to","on top of","under","over",
                      "in front of","near","inside","outside","between"}
    _ACTION_WORDS  = {"riding","holding","carrying","eating","drinking",
                      "wearing","pushing","pulling","walking","running",
                      "sitting","standing","playing","using","throwing",
                      "catching","carrying","holding","driving","leading"}

    # Use str.replace to avoid KeyError if caption contains { or }
    prompt = TRIPLE_EXTRACT_PROMPT.replace("{caption}", caption)

    raw = llm_call(
        [{"role": "user", "content": prompt}],
        max_tokens=600,
    )
    if not raw:
        print(f"    [extract_triples] llm_call returned empty for caption: {caption[:80]!r}")
        return []

    # Strip markdown fences
    raw_stripped = re.sub(r'^```json\s*', '', raw.strip(), flags=re.MULTILINE)
    raw_stripped = re.sub(r'```\s*$', '', raw_stripped.strip(), flags=re.MULTILINE)
    raw_stripped = raw_stripped.strip()

    print(f"    [extract_triples] raw={raw_stripped[:200]!r}")

    try:
        parsed = json.loads(raw_stripped)

        # Handle Llama wrapping output: {"triples": [...]} or {"relations": [...]}
        if isinstance(parsed, dict):
            for key in ("triples", "relations", "result", "data", "output"):
                if key in parsed and isinstance(parsed[key], list):
                    parsed = parsed[key]
                    print(f"    [extract_triples] unwrapped dict key '{key}'")
                    break
            else:
                print(f"    [extract_triples] got dict but no list key: {list(parsed.keys())}")
                return []

        if not isinstance(parsed, list):
            print(f"    [extract_triples] unexpected type: {type(parsed)}")
            return []

        result = []
        for t in parsed:
            if not isinstance(t, dict):
                continue
            if not all(k in t for k in ("subject", "relation", "object")):
                print(f"    [extract_triples] skipping malformed triple: {t}")
                continue

            # Normalise type: accept any capitalisation, infer if missing
            typ = str(t.get("type", "")).upper().strip()
            if typ not in ("SPATIAL", "ACTION", "ATTRIBUTE"):
                rel_lower = t.get("relation", "").lower()
                if any(w in rel_lower for w in _SPATIAL_WORDS):
                    typ = "SPATIAL"
                elif any(w in rel_lower for w in _ACTION_WORDS):
                    typ = "ACTION"
                else:
                    typ = "ACTION"   # safe default: verified by VQA

            t["type"] = typ
            result.append(t)

        return result

    except json.JSONDecodeError as e:
        print(f"    [extract_triples] JSON parse error: {e} | raw={raw_stripped[:300]!r}")
    except Exception as e:
        print(f"    [extract_triples] unexpected error: {e}")
    return []


def _find_best_bbox(entity_name, kb):
    """
    Return the highest-confidence GroundingDINO bbox for entity_name.

    Multi-pass matching to handle descriptive entity phrases from LLaVA/Qwen
    (e.g. "young man", "red bicycle", "the woman on the left"):
      Pass 1: exact core noun   → _ENTITY_SYNONYMS lookup
      Pass 2: each word in core → _ENTITY_SYNONYMS lookup (handles adjective+noun)
      Pass 3: substring match   → any synonym appears inside entity or vice versa

    Returns [x1, y1, x2, y2] normalised, or None if not found.
    """
    core = _core_noun(entity_name)
    if not core:
        return None

    detections = kb.get("detections", [])
    if not detections:
        return None

    def _candidate_synonyms(name):
        """All synonyms for a name, plus the name itself."""
        syns = set(_ENTITY_SYNONYMS.get(name, []) + [name])
        # Also try looking up each individual word
        for w in name.split():
            syns.update(_ENTITY_SYNONYMS.get(w, [w]) + [w])
        return syns

    core_syns = _candidate_synonyms(core)

    best_score, best_box = -1, None
    for label, score, box in detections:
        label_core = _core_noun(label)
        label_syns = _candidate_synonyms(label_core)

        matched = False
        # Pass 1+2: synonym set intersection
        if core_syns & label_syns:
            matched = True
        # Pass 3: substring — "young man" contains "man" which is in person synonyms
        if not matched:
            for cs in core_syns:
                for ls in label_syns:
                    if cs in ls or ls in cs:
                        matched = True
                        break
                if matched:
                    break

        if matched and score > best_score:
            best_score, best_box = score, box

    return best_box


def _crop_to_bboxes(pil_image, box1, box2, padding=0.15):
    """
    Crop PIL image to the union of two normalised bboxes with padding.
    boxes are [x1, y1, x2, y2] in [0, 1] range.
    """
    W, H = pil_image.size
    xs = [box1[0], box1[2], box2[0], box2[2]]
    ys = [box1[1], box1[3], box2[1], box2[3]]
    x1 = max(0.0, min(xs) - padding)
    y1 = max(0.0, min(ys) - padding)
    x2 = min(1.0, max(xs) + padding)
    y2 = min(1.0, max(ys) + padding)
    left, top, right, bottom = int(x1*W), int(y1*H), int(x2*W), int(y2*H)
    # Ensure minimum crop size
    if right - left < 32 or bottom - top < 32:
        return pil_image  # fall back to full image
    return pil_image.crop((left, top, right, bottom))


def _consensus_confirms_triple(subj, rel, obj, cross_captions):
    """
    Returns True if ALL available cross-captioners mention (subj rel obj)
    in their caption, indicating the claim is almost certainly correct.
    Requires at least 2 cross-captioner captions.

    Uses proximity heuristic: the relation verb must appear within 80 chars
    of the subject OR object in the other caption.  Synonym-aware.

    If consensus confirms → skip VQA entirely (free signal, zero API cost).
    """
    if not cross_captions or len(cross_captions) < 2:
        return False

    core_s   = _core_noun(subj)
    core_o   = _core_noun(obj)
    rel_lower = rel.lower()

    confirmed = 0
    for cap_text in cross_captions.values():
        if not cap_text:
            continue
        cap_lower = cap_text.lower()
        if rel_lower not in cap_lower:
            continue
        # Walk every occurrence of the relation verb
        idx = cap_lower.find(rel_lower)
        while idx >= 0:
            context = cap_lower[max(0, idx - 80): idx + 80]
            subj_near = (core_s in context or
                         any(s in context for s in _ENTITY_SYNONYMS.get(core_s, [])))
            obj_near  = (core_o in context or
                         any(s in context for s in _ENTITY_SYNONYMS.get(core_o, [])))
            if subj_near and obj_near:
                confirmed += 1
                break
            idx = cap_lower.find(rel_lower, idx + 1)

    # ALL cross-captioners must confirm (not just majority) to skip VQA
    return confirmed >= len(cross_captions)


def _verify_action_triple(subj, rel, obj, kb, pil_image, n_questions=3):
    """
    Verify an action/attribute triple using crop-based focused VQA
    with multi-question voting (default: 3 paraphrased questions).

    1. Locate subject and object bboxes in GroundingDINO detections.
    2. Crop image to their union (+ padding) to remove background noise.
    3. Ask Maverick n_questions paraphrased YES/NO questions; vote.

    Paraphrasing reduces noise from single-call VQA (a correct claim
    has ~25% chance of being called NO on a single call; with 3 calls
    and majority vote, false-positive rate drops to ~7%).

    Intentionally neutral — no geometry hints are passed here.
    VQA and geometry are independent signals.

    Returns True  (verified correct: majority YES),
            False (hallucination: majority NO),
            None  (cannot verify — missing bbox, API failure, or tie).
    """
    subj_box = _find_best_bbox(subj, kb)
    obj_box  = _find_best_bbox(obj,  kb)

    # When one or both entities have no GDino bbox (common for food, tableware,
    # small objects), fall back to full-image VQA instead of returning None.
    # Crop VQA is more precise; full-image VQA is noisier but still useful.
    using_crop = bool(subj_box and obj_box)
    if using_crop:
        crop = _crop_to_bboxes(pil_image, subj_box, obj_box, padding=0.15)
        buf  = BytesIO()
        crop.convert("RGB").save(buf, format="JPEG", quality=85)
        region_hint = "this cropped region"
    else:
        buf = BytesIO()
        pil_image.convert("RGB").save(buf, format="JPEG", quality=85)
        region_hint = "the full image"
    crop_b64 = base64.b64encode(buf.getvalue()).decode()

    # Multi-strategy VQA: 2 standard YES/NO + 1 contrastive forced-choice.
    # The YES/NO questions establish a baseline; the contrastive question
    # eliminates yes-bias by forcing the VLM to choose between the claimed
    # description and a plausible alternative (counterfactual).
    #
    # Contrastive VQA inspired by Winoground (Thrush et al., CVPR 2022)
    # and VisMin (Awal et al., NeurIPS 2024) compositional reasoning tasks.

    # Generate a counterfactual: negate/alter the relation
    _COUNTERFACTUAL_MAP = {
        "riding": "standing next to", "sitting on": "standing near",
        "holding": "standing next to", "carrying": "walking away from",
        "wearing": "next to", "eating": "looking at",
        "pulling": "pushing", "pushing": "pulling",
        "throwing": "holding", "catching": "dropping",
        "driving": "standing near", "leading": "following",
        "playing with": "ignoring", "using": "near",
        "walking": "standing", "running": "standing",
        "standing on": "next to", "lying on": "sitting near",
        "hanging from": "standing near", "leaning on": "standing near",
    }
    counterfactual_rel = _COUNTERFACTUAL_MAP.get(rel.lower(), f"not {rel}")
    # Randomize A/B order to neutralize position bias.
    # _ab_flip=True → A=counterfactual, B=original.
    # _ab_flip=False → A=original, B=counterfactual (default intuitive order).
    # Deterministic per triple so reruns are consistent.
    _ab_flip = (hash(f"{subj}{rel}{obj}") % 2 == 1)
    if _ab_flip:
        _opt_a, _opt_b = counterfactual_rel, rel
    else:
        _opt_a, _opt_b = rel, counterfactual_rel
    contrastive_q = (
        f'Look at {region_hint} carefully. Which description is more accurate?\n'
        f'(A) The {subj} is {_opt_a} the {obj}\n'
        f'(B) The {subj} is {_opt_b} the {obj}\n'
        f'Answer with ONLY the letter A or B.'
    )

    question_templates = [
        f'In this image, is the {subj} {rel} the {obj}? '
        f'Look carefully at {region_hint}. Answer only YES or NO.',
        f'Does the {subj} appear to be {rel} the {obj} here? '
        f'Observe {region_hint} closely. Answer YES or NO only.',
    ]
    questions_yn = question_templates[:min(n_questions - 1, 2)]  # leave 1 slot for contrastive

    yes_votes = 0
    no_votes  = 0

    # Standard YES/NO questions
    for q in questions_yn:
        result = vlm_call([{"role": "user", "content": [
            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{crop_b64}"}},
            {"type": "text", "text": q},
        ]}], max_tokens=5)
        if not result:
            continue
        r = result.strip().lower()
        if "yes" in r and "no" not in r:
            yes_votes += 1
        elif "no" in r:
            no_votes  += 1

    # Contrastive forced-choice question — uses MAVERICK_VLM for stronger signal.
    # Maverick is a 17B MoE model; its forced-choice answers are more reliable
    # than Qwen3-VL-8B on compositional relations. Combined with A/B randomization,
    # this makes the contrastive vote the highest-quality signal in the pipeline.
    contrastive_no = False   # track separately for confidence gate
    contrastive_result = vlm_call([{"role": "user", "content": [
        {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{crop_b64}"}},
        {"type": "text", "text": contrastive_q},
    ]}], max_tokens=5, model=MAVERICK_VLM)
    if contrastive_result:
        cr    = contrastive_result.strip().upper()
        chose_a = ("A" in cr and "B" not in cr)
        chose_b = ("B" in cr and "A" not in cr)
        if _ab_flip:
            # A=counterfactual, B=original
            if chose_b:
                yes_votes += 1            # chose original → claim correct
            elif chose_a:
                no_votes    += 1
                contrastive_no = True     # Maverick: chose counterfactual → hallucination
        else:
            # A=original, B=counterfactual (default)
            if chose_a:
                yes_votes += 1            # chose original → claim correct
            elif chose_b:
                no_votes    += 1
                contrastive_no = True     # Maverick: chose counterfactual → hallucination

    total = yes_votes + no_votes
    if total == 0:
        result = None  # all calls failed or ambiguous
    elif yes_votes > no_votes:
        result = True   # majority YES → claim verified correct
    elif no_votes > yes_votes:
        result = False  # majority NO  → hallucination confirmed
    else:
        result = None   # exact tie → uncertain

    # Return 5-tuple: (verdict, yes_votes, no_votes, total, contrastive_no)
    # contrastive_no=True means Maverick's debiased forced-choice voted NO.
    # Callers use this for the "contrastive_NO + 1 standard_NO = HIGH" gate.
    return (result, yes_votes, no_votes, total, contrastive_no)


def _apply_triple_correction(caption, wrong_phrase, correct_phrase,
                              subj="", obj_=""):
    """
    Fix exactly one relationship word/phrase in the caption.

    wrong_phrase: the extracted relation verb (e.g. "riding")
    correct_phrase: the correct relation (e.g. "standing next to")
    subj, obj_: entity names for context (used in fast path and Llama prompt)

    Fast path: verb found verbatim → find occurrence nearest to subj/obj
               mentions to avoid replacing the wrong instance.
    Llama path: verb not verbatim → Llama finds the right phrase by context.
    """
    cap_lower = caption.lower()
    wp_lower  = wrong_phrase.lower()

    if wp_lower in cap_lower:
        # Fast path — but find the occurrence NEAREST to the subject/object
        # to avoid replacing the wrong instance in multi-entity captions.
        # e.g. "man holding child ... man holding bicycle" — fix (man, bicycle)
        # by finding the "holding" closest to "bicycle".
        occurrences = []
        start = 0
        while True:
            idx = cap_lower.find(wp_lower, start)
            if idx == -1:
                break
            occurrences.append(idx)
            start = idx + 1

        if len(occurrences) == 1:
            idx = occurrences[0]
        else:
            # Score each occurrence by proximity to subj + obj mentions
            subj_idx = cap_lower.find(_core_noun(subj)) if subj else -1
            obj_idx  = cap_lower.find(_core_noun(obj_)) if obj_ else -1
            def _proximity(i):
                d = 0
                if subj_idx >= 0:
                    d += abs(i - subj_idx)
                if obj_idx >= 0:
                    d += abs(i - obj_idx)
                return d
            idx = min(occurrences, key=_proximity)

        return caption[:idx] + correct_phrase + caption[idx + len(wrong_phrase):]

    # Llama path: relation verb not verbatim — describe by entity context.
    # Ratio guard is looser here (1.25) because replacing a short verb with
    # a multi-word phrase legitimately lengthens the caption.
    raw = llm_call([{"role": "user", "content": TRIPLE_CORRECT_PROMPT.format(
        caption=caption,
        subj=subj or wrong_phrase,
        obj=obj_ or correct_phrase,
        wrong_phrase=wrong_phrase,
        correct_phrase=correct_phrase,
    )}], max_tokens=int(len(caption.split()) * 2.5))

    if raw:
        raw = raw.strip().strip('"').strip("'")
        ratio = len(raw) / max(len(caption), 1)
        # Allow up to 25% growth (multi-word replacements) but reject compression
        if 0.85 <= ratio <= 1.25:
            return raw

    return caption  # fall back to original if correction fails


# ── Main v2 correction function ───────────────────────────────────────────


MISSING_FACTS_PROMPT = """You are improving an image caption by inserting missing facts at semantically appropriate positions.

CAPTION:
"{caption}"

VISUAL DESCRIPTION (from a separate vision model — treat as ground truth):
"{visual_description}"

TASK:
1. Identify up to 2 important facts visible in the Visual Description but COMPLETELY ABSENT from the Caption.
   Missing facts can be anything: objects, actions, attributes, colors, counts, materials, spatial relationships, context.

   Priority order (highest value first):
     A. MISSING OBJECTS — an entire object or entity not mentioned at all in the caption.
        When adding a missing object, include WHERE it is relative to the main scene naturally
        (e.g. "a wooden box resting on the windowsill above the sink").
        This is the most valuable type of insertion because it answers relational questions.
     B. MISSING ACTIONS — what a person or animal is doing, if not described.
     C. MISSING ATTRIBUTES — important attributes (count, color, material) of objects already in caption.

   Skip: vague background details, minor clutter, anything already implied or easily inferred.

2. For each missing fact, find the sentence in the caption it belongs with and insert it there naturally.
   The right sentence is the one that already discusses the same subject or scene context.
   - Missing object → insert in the most scene-descriptive sentence, or append a short sentence
   - Missing action → insert in the sentence describing what that entity is doing
   - Missing attribute → insert in the sentence describing that entity
   - If truly no sentence fits, append one short sentence at the end

RULES:
- Only INSERT — never delete, rephrase, or shorten any existing text.
- Insert BETWEEN complete clauses or at sentence boundaries — NEVER inside an existing noun phrase or adjective phrase.
  Wrong: "lettering on the knee the right knee"  ← inserted inside "on the right knee"
  Right: "lettering on the right knee, and he is also wearing X"  ← appended after
- Use natural conjunctions ("and", "also", "along with", "with", ", including") to weave in the new fact.
- When introducing a missing object, include its location relative to other objects naturally.
- If nothing important is missing: output exactly NONE.

Output the full modified caption ONLY. No explanation, no prefix."""


def _add_missing_fact_addendum(corrected_caption, kb):
    """
    Insert facts from Maverick's VLM description that are absent from the long caption,
    placing them at the most natural position rather than blindly appending.
    Zero extra API cost — Maverick description was computed in KB-construction step.

    Returns (new_caption, n_modified) where n_modified is 0 or 1.
    """
    visual_desc = kb.get("visual_description", "")
    if not visual_desc or len(visual_desc.strip()) < 20:
        return corrected_caption, 0

    # Skip very long captions (> 300 words) — noise risk outweighs benefit
    if len(corrected_caption.split()) > 300:
        return corrected_caption, 0

    prompt = MISSING_FACTS_PROMPT.format(
        caption=corrected_caption[:1500],
        visual_description=visual_desc[:1500],
    )
    raw = llm_call([{"role": "user", "content": prompt}],
                   model=LLM_MODEL, max_tokens=800)
    if not raw:
        return corrected_caption, 0

    result = raw.strip().strip('"').strip("'").strip()

    # If model says nothing is missing
    if result.upper() == "NONE" or not result:
        return corrected_caption, 0

    # Safety: result must be longer than original (only insertions allowed)
    # and not too much longer (cap at 60 extra words to prevent hallucination drift)
    orig_words = len(corrected_caption.split())
    new_words  = len(result.split())
    added = new_words - orig_words

    if added <= 0:
        # Model shortened the caption — rejected (rewrite, not insertion)
        return corrected_caption, 0
    if added > 30:
        # Too many new words — likely rewrote rather than inserted
        return corrected_caption, 0

    # Sanity: at least 80% of original caption words must survive
    orig_lower = corrected_caption.lower()
    result_lower = result.lower()
    orig_tokens = orig_lower.split()
    surviving = sum(1 for t in orig_tokens if t in result_lower)
    if surviving / max(len(orig_tokens), 1) < 0.80:
        return corrected_caption, 0

    # Repetition guard: reject if the LLM inserted a phrase INSIDE an existing
    # phrase, creating artifacts like "on the knee the right knee."
    # Two checks:
    #   A. Any bigram/trigram appears twice within a 7-token window.
    #   B. Any content word (not stopword) appears twice within a 5-token window
    #      in the result but only once in the original — insertion duplicate.
    _STOPWORDS = {"a","an","the","and","or","but","in","on","at","to","of",
                  "is","are","was","were","be","been","being","with","by",
                  "for","from","that","this","it","its","he","she","they"}
    result_toks = result.lower().split()
    orig_toks   = corrected_caption.lower().split()

    # Check A: repeated n-gram within window
    for i in range(len(result_toks) - 5):
        window = result_toks[i:i+7]
        for span in (2, 3):
            gram = tuple(window[:span])
            rest = window[1:]
            for j in range(len(rest) - span + 1):
                if tuple(rest[j:j+span]) == gram:
                    return corrected_caption, 0

    # Check B: any bigram that is NEW in the result (not in original) appears
    # twice — sign that the LLM duplicated a phrase during insertion.
    orig_bigrams = set(zip(orig_toks, orig_toks[1:]))
    result_bigrams = [tuple(pair) for pair in zip(result_toks, result_toks[1:])]
    for bg in result_bigrams:
        if bg not in orig_bigrams and result_bigrams.count(bg) >= 2:
            return corrected_caption, 0  # duplicated new bigram

    return result, 1



def _correct_long_caption_v2(img_id, caption, kb, pil_image, cross_captions=None):
    """
    Per-triple verification correction for detailed captions (>= 30 words).

    Architecture:
      1. Llama extracts (subject, relation, object, type) triples from caption
      2. Consensus pre-filter: if ALL cross-captioners confirm the claim, skip VQA
      3. SPATIAL triples  → verified by deterministic bbox geometry
         ACTION triples   → verified by 3-question majority-vote crop VQA (Maverick)
         ATTRIBUTE triples→ verified by 3-question majority-vote crop VQA
      4. Confirmed-incorrect triples corrected one at a time via Llama
      5. Spatial addendum appended

    Key properties:
      - Consensus filter (free, zero API cost): skip VQA for claims confirmed by
        both BLIP-2 and LLaVA/Qwen → reduces false positives on strong captioners
      - Multi-question voting (3 paraphrases, majority): false-positive rate drops
        from ~25% (single call) to ~7% per triple
    """
    if pil_image is None:
        return {
            "caption": caption, "corrected": caption,
            "errors": [], "missing": [], "edit_rate": 0,
            "status": "unchanged", "mode": "correct_v2",
        }

    # Step 1: extract triples
    triples = _extract_triples(caption)
    if not triples:
        # Fallback: addendum only if triple extraction failed
        print(f"    [{img_id}] triple extraction returned 0 triples — addendum only")
        corrected, n_addendum = _build_spatial_addendum(caption, kb)
        return {
            "caption": caption, "corrected": corrected,
            "errors": [], "missing": [], "edit_rate": levenshtein(caption, corrected) / max(len(caption), 1),
            "status": "modified" if corrected != caption else "unchanged",
            "mode": "correct_v2", "n_triples": 0, "n_addendum": n_addendum,
        }

    spatial_facts  = kb.get("spatial_facts", [])
    geo_contras    = _check_spatial_contradictions(caption, spatial_facts)
    geo_set        = {c.lower() for c in geo_contras}

    errors       = []   # confirmed hallucinations
    all_checks   = []   # full audit trail

    for t in triples:
        subj = t["subject"].strip()
        rel  = t["relation"].strip()
        obj  = t["object"].strip()
        typ  = t["type"].upper()
        claim_str = f"{subj} {rel} {obj}"

        if typ == "SPATIAL":
            # Deterministic: check geometry
            kb_triples = _parse_spatial_facts(spatial_facts)
            verdict, confidence, reason = "UNKNOWN", "LOW", "no geometry available"
            for kb_s, kb_r, kb_o in kb_triples:
                if _entity_matches(subj, kb_s) and _entity_matches(obj, kb_o):
                    if kb_r.lower() in _spatial_synonyms(rel):
                        verdict, confidence, reason = "CORRECT", "HIGH", "geometry confirms"
                        break
                    opp = SPATIAL_OPPOSITES.get(rel)
                    if opp and kb_r == opp:
                        verdict, confidence, reason = "INCORRECT", "HIGH", f"geometry shows {kb_r}"
                        break
            # Also check pre-computed geometry contradictions
            if any(claim_str.lower() in g or
                   (subj.lower() in g and rel.lower() in g) for g in geo_set):
                verdict, confidence, reason = "INCORRECT", "HIGH", "deterministic geometry contradiction"
            # VQA fallback for SPATIAL UNKNOWN:
            # (a) Object not detected by GDino → existence VQA
            # (b) Object detected but no geometry rule matched → relation VQA
            #     If VQA says relation is wrong → ask a second "what IS the
            #     correct relation?" question so the corrector has ground truth.
            # Maverick VQA confirmation for geometry INCORRECT.
            # GDino geometry is fast but error-prone (wrong labels, imprecise bboxes).
            # Before trusting a HIGH-confidence geometry contradiction, ask Maverick
            # to confirm. If Maverick disagrees → downgrade to UNKNOWN and skip.
            # This eliminates geometry false positives (e.g. lamp "on table" → "below").
            if verdict == "INCORRECT" and confidence == "HIGH":
                vqa_confirm, _, _, _, _ = _verify_action_triple(subj, rel, obj, kb, pil_image, n_questions=2)
                if vqa_confirm is True:
                    # Maverick says the claimed relation IS correct → geometry wrong
                    verdict    = "CORRECT"
                    confidence = "MEDIUM"
                    reason     = f"geometry said INCORRECT but Maverick VQA confirmed → kept"
                elif vqa_confirm is None:
                    # Maverick uncertain → don't act on geometry alone
                    verdict    = "UNKNOWN"
                    confidence = "LOW"
                    reason     = "geometry INCORRECT but Maverick uncertain → skipped"
                # vqa_confirm is False → both agree → keep HIGH INCORRECT

            if verdict == "UNKNOWN":
                obj_box_check = _find_best_bbox(obj, kb)
                if obj_box_check is None:
                    # Object not detected — ask if it even exists in the image
                    exists = _check_entity_exists_vqa(obj, pil_image)
                    if exists is False:
                        verdict    = "INCORRECT"
                        confidence = "MEDIUM"
                        reason     = f"object '{obj}' not detected; VQA confirms absence"
                    else:
                        # Object detected by existence VQA but no GDino bbox.
                        # GDino commonly misses food, tableware, small objects.
                        # Fall back to full-image VQA — _verify_action_triple will
                        # use the full image when bboxes are unavailable.
                        vqa_result, _, _, _, _ = _verify_action_triple(subj, rel, obj, kb, pil_image, n_questions=3)
                        if vqa_result is False:
                            correct_spatial = _query_correct_spatial_relation(
                                subj, obj, kb, pil_image
                            )
                            verdict    = "INCORRECT"
                            confidence = "MEDIUM"
                            reason     = (f"spatial VQA (full-image, no GDino bbox) rejected; "
                                          f"correct relation: '{correct_spatial}'"
                                          if correct_spatial else
                                          "spatial VQA (full-image, no GDino bbox) rejected")
                        elif vqa_result is True:
                            verdict    = "CORRECT"
                            confidence = "MEDIUM"
                            reason     = "spatial VQA (full-image, no GDino bbox) confirmed"
                        # vqa_result is None → truly uncertain → leave UNKNOWN
                else:
                    # Both entities detectable and geometry had no match.
                    # Step 1: verify the claimed relation with Maverick VQA
                    vqa_result, _, _, _, _ = _verify_action_triple(subj, rel, obj, kb, pil_image, n_questions=3)
                    if vqa_result is False:
                        # Step 2: ask Maverick what the CORRECT spatial relation is
                        correct_spatial = _query_correct_spatial_relation(
                            subj, obj, kb, pil_image
                        )
                        verdict    = "INCORRECT"
                        confidence = "MEDIUM"
                        reason     = (f"spatial VQA rejected; correct relation: '{correct_spatial}'"
                                      if correct_spatial else "spatial VQA fallback rejected")
                    elif vqa_result is True:
                        verdict    = "CORRECT"
                        confidence = "MEDIUM"
                        reason     = "spatial VQA fallback confirmed"

            all_checks.append({"claim": claim_str, "type": typ,
                                "verdict": verdict, "confidence": confidence, "reason": reason})
            if verdict == "INCORRECT" and confidence == "HIGH":
                errors.append({"claim": claim_str, "subject": subj,
                               "relation": rel, "object": obj,
                               "reason": reason, "confidence": confidence,
                               "type": "SPATIAL"})

        else:  # ACTION or ATTRIBUTE
            # Layer 1: action geometry pre-filter (Tier 1 + Tier 2)
            # Tier 1 (mounting, containment, adjacency): bbox-only
            # Tier 2 (grasping, consuming): requires ViTPose keypoints
            # If geometric prerequisite is VIOLATED → confirmed hallucination,
            # skip VQA entirely (saves an API call + higher precision).
            geo_family = _classify_action_family(rel)
            geo_prereq = None
            if geo_family:
                s_box = _find_best_bbox(subj, kb)
                o_box = _find_best_bbox(obj, kb)
                if s_box and o_box:
                    # For Tier 2: get keypoints from the PERSON entity.
                    # Subject isn't always the person ("cup held by man" →
                    # subject=cup). Check which entity is a person/animal.
                    kp = None
                    family_spec = ACTION_GEOMETRY_TAXONOMY.get(geo_family, {})
                    if family_spec.get("needs_keypoints"):
                        _PERSON_WORDS = {"person", "man", "woman", "boy", "girl",
                                         "child", "kid", "baby", "player", "rider",
                                         "worker", "people", "dog", "cat", "horse"}
                        subj_lower = _core_noun(subj)
                        obj_lower  = _core_noun(obj)
                        if subj_lower in _PERSON_WORDS:
                            kp = _get_person_keypoints(pil_image, s_box)
                        elif obj_lower in _PERSON_WORDS:
                            kp = _get_person_keypoints(pil_image, o_box)
                            # Swap perspective: now "person" keypoints are
                            # relative to what was the object bbox, but we
                            # still check wrist/nose vs the OTHER entity's bbox.
                            # So swap s_box/o_box for the geometry check.
                            s_box, o_box = o_box, s_box

                    geo_prereq = _check_action_geometry(
                        geo_family, s_box, o_box, keypoints=kp
                    )
                    if geo_prereq is False:
                        tier = "Tier2-keypoint" if kp else "Tier1-bbox"
                        print(f"    [{img_id}] geometry flag ({geo_family}/{tier}): '{claim_str}' — confirming with VQA")

            # Layer 1.5: cross-captioner consensus pre-filter (zero API cost)
            # If both other captioners also describe "subj rel obj", the claim
            # is almost certainly correct — skip VQA to avoid false positives.
            if cross_captions and _consensus_confirms_triple(subj, rel, obj, cross_captions):
                verdict, confidence, reason = "CORRECT", "HIGH", "cross-captioner consensus"
                all_checks.append({"claim": claim_str, "type": typ,
                                    "verdict": verdict, "confidence": confidence, "reason": reason})
                continue  # no VQA needed

            # Layer 2: crop-based VQA with multi-question voting (Maverick)
            # Runs 3 paraphrased YES/NO questions, majority vote.
            # Geometry FAIL is evidence, not a final verdict — VQA must confirm.
            # IMPORTANT: VQA gets no geometry hints — independence required
            # so that geometry+VQA agreement is a genuinely strong signal.
            verified, v_yes, v_no, v_total, v_contrastive_no = _verify_action_triple(subj, rel, obj, kb, pil_image, n_questions=3)
            unanimous_no = (verified is False and v_no == v_total and v_total >= 2)
            # New: Maverick contrastive vote + at least 1 standard NO = HIGH.
            # std_no = standard YES/NO NO-votes (exclude the contrastive vote).
            std_no = (v_no - 1) if v_contrastive_no else v_no
            contrastive_high = v_contrastive_no and std_no >= 1
            print(f"    [{img_id}] {typ} triple ({subj!r},{rel!r},{obj!r}) "
                  f"geo={geo_prereq} vqa={verified} votes={v_yes}Y/{v_no}N/{v_total}T")

            if verified is True:
                verdict, confidence, reason = "CORRECT", "MEDIUM", "crop VQA confirmed"
            elif verified is False:
                if geo_prereq is False:
                    # Both geometry AND VQA agree it's wrong — high confidence
                    tier = "Tier2-keypoint" if kp else "Tier1-bbox"
                    verdict    = "INCORRECT"
                    confidence = "HIGH"
                    reason     = f"geometry ({geo_family}/{tier}) + VQA both FAILED"
                elif unanimous_no:
                    verdict    = "INCORRECT"
                    confidence = "HIGH"
                    reason     = f"unanimous VQA rejection ({v_no}/{v_total} NO)"
                elif contrastive_high:
                    # Maverick contrastive (debiased A/B) + ≥1 standard NO.
                    # Two independent signals agree → treat as HIGH.
                    # This is the key recall-improvement path: catches hallucinations
                    # that would otherwise be MEDIUM (split vote, no geometry).
                    verdict    = "INCORRECT"
                    confidence = "HIGH"
                    reason     = f"Maverick contrastive NO + {std_no} standard NO ({v_no}/{v_total} total)"
                else:
                    # Split VQA without contrastive confirmation → MEDIUM → skipped
                    verdict    = "INCORRECT"
                    confidence = "MEDIUM"
                    reason     = f"split VQA rejection ({v_no}/{v_total} NO, no contrastive confirm)"
                if confidence == "HIGH":
                    errors.append({"claim": claim_str, "subject": subj,
                               "relation": rel, "object": obj,
                               "reason": reason, "confidence": confidence,
                               "type": typ})
            else:
                if geo_prereq is False:
                    # Geometry said FAIL but VQA is uncertain/says OK
                    # → geometry was likely wrong (perspective, occlusion, etc.)
                    verdict    = "UNKNOWN"
                    confidence = "LOW"
                    reason     = "geometry flagged but VQA inconclusive — skipping"
                else:
                    verdict, confidence, reason = "UNKNOWN", "LOW", "could not verify (missing bbox or uncertain)"
            all_checks.append({"claim": claim_str, "type": typ,
                                "verdict": verdict, "confidence": confidence, "reason": reason})

    # Step 3: batched correction — Llama fixes ALL confirmed hallucinations in one call.
    # This avoids the surgical-edit garbling problem (dangling "it up", "pink mat it")
    # and handles full-sentence removal (e.g. "The treadmill is located in the center").
    corrected = caption
    applied   = []

    if errors:
        # Build numbered error list with evidence for each hallucination
        error_lines = []
        for i, err in enumerate(errors, 1):
            subj, rel, obj_ = err["subject"], err["relation"], err["object"]
            reason = err["reason"]
            err_type = err.get("type", "ACTION")
            # Give Llama different guidance depending on type
            if err_type == "SPATIAL" and "absence" in reason:
                guidance = f"The {obj_} is NOT present in the image — remove this claim entirely."
            elif err_type == "SPATIAL":
                # Extract ground-truth relation from reason string if available
                # (_query_correct_spatial_relation embeds it as "correct relation: 'X'")
                import re as _re
                _m = _re.search(r"correct relation[:\s]+[\'\"](.*?)[\'\"]", reason)
                correct_rel = _m.group(1) if _m else SPATIAL_OPPOSITES.get(rel)
                if correct_rel:
                    guidance = f"The correct spatial relation is '{correct_rel}' — replace '{rel}' with '{correct_rel}'."
                else:
                    guidance = f"This spatial claim is incorrect — remove or soften it."
            else:
                # Action / attribute: give confidence-aware actionable guidance
                # (we don't know the correct relation, so tell Llama how to handle it)
                if confidence == "HIGH":
                    guidance = (
                        f"The claim that '{subj} {rel} {obj_}' is DEFINITELY WRONG "
                        f"(VQA confidence HIGH — {reason}). "
                        f"Remove the sentence(s) describing this, or if embedded in a "
                        f"longer sentence, rephrase so the false claim is absent."
                    )
                else:
                    guidance = (
                        f"The claim that '{subj} {rel} {obj_}' appears incorrect "
                        f"(VQA confidence MEDIUM — {reason}). "
                        f"Replace or soften the '{rel}' relationship between "
                        f"'{subj}' and '{obj_}' to something more accurate."
                    )"
            error_lines.append(f'{i}. "{subj} {rel} {obj_}" — {guidance}')

        error_list_str = "\n".join(error_lines)
        prompt = BATCH_CORRECT_PROMPT.format(caption=caption, error_list=error_list_str)

        raw = llm_call(
            [{"role": "user", "content": prompt}],
            max_tokens=int(len(caption.split()) * 2.5 + 50),
        )

        if raw:
            candidate = raw.strip().strip('"').strip("'")
            orig_len  = len(caption)
            cand_len  = len(candidate)
            ratio     = cand_len / max(orig_len, 1)

            # Fluency gate:
            # 1. Length (asymmetric): no minimum — false sentences may be removed entirely;
            #    cap upward at 1.30 to prevent LLM adding new content.
            # 2. Must differ from original (something changed)
            # 3. Must have at least 5 words (not a blank/summary)
            # 4. Garble check: no artifacts from character-level insertion
            # Garble check: look for artifacts from bad character-level insertion.
            # Use simple substring checks — avoids \b word boundary bytes
            # that corrupt the notebook source when embedded as control chars.
            def _has_garble(text):
                t = text.lower()
                # "pulling a sled it up" pattern: word followed by "it up"
                if re.search(r'\w+\s+it\s+up', t): return True
                # "on pink mat it" pattern: "mat it" fragment
                if "mat it" in t: return True
                # extremely long merged word (no spaces in 30+ char run)
                if re.search(r'\S{30,}', t): return True
                return False

            # Partial-replacement contradiction check.
            # The batch LLM sometimes replaces word W with W' in one sentence
            # but misses other sentences that use W — leaving W and W' coexisting.
            # This makes the caption worse (e.g. "pushing... actively pulling").
            #
            # Two complementary checks:
            #   A. Structural: any content word that was REDUCED in count (but
            #      not fully removed) while a new content word appeared → partial
            #      replacement. Only flag verbs, not adjectives (adjectives like
            #      "blue" can legitimately apply to different objects).
            #   B. Known-opposites: use existing SPATIAL_OPPOSITES dict + common
            #      action antonyms to catch the most damaging cases.
            from collections import Counter as _Ctr
            import re as _re2

            _VERB_PATTERN = _re2.compile(
                r'(pull|push|lift|lower|open|clos|enter|exit|stand|sit|'
                r'hold|release|give|tak|carry|drop|lead|follow|ascend|descend|'
                r'mov|turn|rais|lower)\w*', _re2.IGNORECASE
            )
            _ALL_OPP = {**SPATIAL_OPPOSITES}  # spatial opposites already defined
            _ALL_OPP.update({                  # add common action antonyms
                "pull": "push", "push": "pull",
                "pulling": "pushing", "pushing": "pulling",
                "pulled": "pushed", "pushed": "pulled",
                "lift": "lower", "lifting": "lowering", "lifted": "lowered",
                "lower": "lift", "lowering": "lifting", "lowered": "lifted",
                "open": "close", "opening": "closing", "opened": "closed",
                "close": "open", "closing": "opening", "closed": "opened",
                "enter": "exit", "entering": "exiting", "entered": "exited",
                "exit": "enter", "exiting": "entering", "exited": "entered",
                "stand": "sit", "standing": "sitting", "stood": "sat",
                "sit": "stand", "sitting": "standing",
                "ascend": "descend", "ascending": "descending",
                "descend": "ascend", "descending": "ascending",
            })

            def _has_self_contradiction(original_cap, corrected_cap):
                """
                Two complementary checks — both require an opposite to appear,
                so valid removals (e.g. drop one 'lifting' sentence) are not flagged.

                Check A (structural): a verb W was PARTIALLY replaced (count reduced
                  but not to zero) AND its known opposite appears in the corrected
                  caption → the replacement was only applied in some sentences.
                  e.g. "pulling" → "pushing" in sentence 1, "pulling" still in sentence 2.

                Check B (known-opposites): any word from _ALL_OPP and its opposite
                  are both present in the corrected caption.
                  e.g. "above" and "below" both appear describing the same pair.
                """
                orig_words = original_cap.lower().split()
                orig_cnt   = _Ctr(orig_words)
                corr_lower = corrected_cap.lower()
                corr_words = corr_lower.split()
                corr_cnt   = _Ctr(corr_words)

                for word in set(orig_words):
                    if len(word) < 4:
                        continue
                    opp = _ALL_OPP.get(word)
                    if opp is None:
                        continue
                    o_c = orig_cnt[word]
                    c_c = corr_cnt.get(word, 0)

                    # Check A: partial replacement — word still present but count shrank,
                    #           AND its opposite now appears (= replacement applied somewhere)
                    if 0 < c_c < o_c and opp in corr_lower:
                        return True

                    # Check B: both word and its opposite now present (regardless of original)
                    if word in corr_lower and opp in corr_lower:
                        return True

                return False

            has_garble       = _has_garble(candidate)
            has_contradiction = _has_self_contradiction(caption, candidate)

            # No lower bound — removing entire false sentences is expected and valid.
            # Cap upward at 1.30 to prevent LLM padding new content.
            is_too_short = len(candidate.split()) < 5
            if ratio <= 1.30 and candidate != caption and not has_garble and not has_contradiction and not is_too_short:
                corrected = candidate
                applied = [{"method": "batch_llm", "n_errors": len(errors),
                             "errors": [e["claim"] for e in errors]}]
                print(f"    [v2] batch correction: {len(errors)} errors fixed "
                      f"(len {orig_len}→{cand_len}, ratio={ratio:.2f})")
            else:
                print(f"    [v2] batch correction REJECTED "
                      f"(ratio={ratio:.2f}, garble={has_garble}, same={candidate==caption})")
                # Fallback: try applying only the HIGHEST confidence errors one at a time
                high_errors = [e for e in errors if e.get("confidence") == "HIGH"]
                for err in high_errors:
                    subj, rel, obj_ = err["subject"], err["relation"], err["object"]
                    err_type = err.get("type", "ACTION")
                    if err_type == "SPATIAL":
                        correct_rel = SPATIAL_OPPOSITES.get(rel, rel)
                        if correct_rel != rel:
                            prev = corrected
                            corrected = _apply_triple_correction(corrected, rel, correct_rel, subj, obj_)
                            if corrected != prev:
                                applied.append({"wrong": rel, "correct": correct_rel,
                                                 "claim": f"{subj} {rel} {obj_}",
                                                 "method": "fallback_spatial"})
                if applied:
                    print(f"    [v2] fallback: applied {len(applied)} HIGH-confidence spatial fix(es)")

    # Step 4: Post-correction verification.
    # Re-extract triples from the corrected caption and check each against the KB.
    # If a corrected triple CONTRADICTS the KB → the LLM introduced a new error.
    # In that case, reject the entire correction and keep the original caption.
    # This closes the loop on "correction hallucinations" at zero extra VLM cost
    # (only uses Llama for triple extraction + deterministic KB checks).
    if corrected != caption:
        new_triples = _extract_triples(corrected)
        introduced_errors = []
        for nt in new_triples:
            ns, nr, no = nt["subject"].strip(), nt["relation"].strip(), nt["object"].strip()
            ntyp = nt["type"].upper()
            if ntyp == "SPATIAL":
                kb_triples = _parse_spatial_facts(kb.get("spatial_facts", []))
                for kb_s, kb_r, kb_o in kb_triples:
                    if _entity_matches(ns, kb_s) and _entity_matches(no, kb_o):
                        opp = SPATIAL_OPPOSITES.get(nr.lower())
                        if opp and kb_r.lower() == opp:
                            introduced_errors.append(f"{ns} {nr} {no} (KB says {kb_r})")
                            break

        if introduced_errors:
            print(f"    [{img_id}] POST-CHECK FAILED: correction introduced new errors: "
                  f"{introduced_errors} → reverting to original")
            corrected = caption
            applied = []

    # Step 5: Verified missing-fact addendum for long captions.
    # Uses the Maverick VLM description already in the KB (zero extra API cost).
    # Only APPENDS facts — never rewrites — so no compression regression risk.
    corrected, n_addendum = _add_missing_fact_addendum(corrected, kb)
    if n_addendum:
        print(f"    [{img_id}] addendum: +{n_addendum} missing fact(s) appended")

    edit_rate = levenshtein(caption, corrected) / max(len(caption), len(corrected), 1)
    return {
        "caption":     caption,
        "corrected":   corrected,
        "errors":      errors,
        "all_checks":  all_checks,
        "applied":     applied,
        "missing":     [],
        "edit_rate":   edit_rate,
        "n_triples":   len(triples),
        "n_addendum":  n_addendum,
        "status":      "modified" if corrected != caption else "unchanged",
        "mode":        "correct_v2",
    }

# -- Unified entry point --------------------------------------------------

In [ ]:
# ============================================================
# Cell 3 — Fast Hallucination Detection Test
# ============================================================
# For each test image:
#   1. Qwen3-VL-8B generates a base 2-3 sentence description.
#   2. A confirmed-false relation (R-Bench answer=NO) is injected.
#   3. _correct_long_caption_v2 runs with EMPTY_KB → full-image VQA.
#   4. We check whether the injected triple was flagged INCORRECT.
#
# No BLIP-2, no GDino, no checkpoints needed.
# ============================================================

import re
from difflib import SequenceMatcher

N_TEST    = 5       # ← set to 10-20 for more coverage
random.seed(42)

# ── R-Bench question parser ──────────────────────────────────────────────
_Q_RE = re.compile(
    r"^Is (?:the |a )?(.+?) (on top of|to the left of|to the right of|"
    r"in front of|at the back of|above|below|behind|beside|next to|near|"
    r"inside|outside|on|under|holding|wearing|riding|carrying|eating|"
    r"using|with|between|across from|facing|attached to|leaning on|"
    r"standing on|sitting on|lying on|hanging from) (?:the |a )?(.+?)\??$",
    re.IGNORECASE
)

def parse_q(question):
    m = _Q_RE.match(question.strip())
    if m:
        return m.group(1).strip(), m.group(2).strip(), m.group(3).strip()
    res = llm_call([{"role": "user", "content":
        f'Extract (subject,relation,object) from: "{question}"\n'
        f'Reply ONLY with JSON: {{"subject":"","relation":"","object":""}}'
    }], max_tokens=60)
    if res:
        m2 = re.search(r'\{[^}]+\}', res)
        if m2:
            try:
                d = json.loads(m2.group())
                if d.get("subject") and d.get("relation") and d.get("object"):
                    return d["subject"], d["relation"], d["object"]
            except: pass
    return None

def hits(claim, s, o):
    d = claim.lower()
    return s.lower() in d and o.lower() in d

# ── Empty KB — GDino not loaded, full-image VQA handles everything ───────
EMPTY_KB = {
    "hard_facts": [], "spatial_facts": [], "geometry_contradictions": [],
    "description": "", "objects": [],
}

# ── Build NO-answer index ────────────────────────────────────────────────
no_by_img = {}
for q in rbench_data:
    if q.get("answer", "") == "no":
        # Support both old format (image key = filename) and normalized (image_id = stem)
        raw_key  = q.get("image_id") or q.get("image", "")
        img_stem = re.sub(r"\.(jpe?g|png)$", "", os.path.basename(str(raw_key)), flags=re.IGNORECASE)
        if img_stem:
            no_by_img.setdefault(img_stem, []).append(q)

candidates = [img_id for img_id in images if img_id in no_by_img]
random.shuffle(candidates)
test_ids = candidates[:N_TEST]
print(f"Testing on {len(test_ids)} images  ({len(candidates)} candidates available)\n")
print("=" * 60)

# ── Main loop ────────────────────────────────────────────────────────────
results = []

for img_id in test_ids:
    pil_img = images[img_id]
    print(f"\n[{img_id}]")

    # Step 1: get base caption via Qwen
    buf = BytesIO()
    pil_img.convert("RGB").save(buf, format="JPEG", quality=85)
    b64 = base64.b64encode(buf.getvalue()).decode()
    base_cap = vlm_call([{"role": "user", "content": [
        {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}"}},
        {"type": "text",
         "text": "Describe what you see in this image in 2-3 sentences. "
                 "Mention objects, their positions, and any actions."},
    ]}], max_tokens=150)
    if not base_cap:
        print("  SKIP: VLM caption failed"); continue
    print(f"  Caption  : {base_cap[:120]}...")

    # Step 2: find parseable R-Bench NO-answer question
    parsed = None
    for q in random.sample(no_by_img[img_id], min(8, len(no_by_img[img_id]))):
        p = parse_q(q["question"])
        if p and p[0] and p[1] and p[2]:
            s, r, o = p
            if s.lower() not in base_cap.lower() or r.lower() not in base_cap.lower():
                parsed = (s, r, o, q["question"])
                break
    if not parsed:
        print("  SKIP: no parseable NO-answer question"); continue
    inj_s, inj_r, inj_o, source_q = parsed

    # Step 3: inject false relation
    inject_sent   = f" Additionally, a {inj_s} is {inj_r} the {inj_o}."
    hallucinated  = base_cap + inject_sent
    print(f"  Injection: '{inj_s} {inj_r} {inj_o}'")
    print(f"  Source Q : {source_q}")

    # Step 4: run detection
    result    = _correct_long_caption_v2(img_id, hallucinated, EMPTY_KB, pil_img)
    all_chks  = result.get("all_checks", [])
    errors    = result.get("errors",     [])

    # Step 5: score
    detected  = any(hits(e["claim"], inj_s, inj_o) for e in errors)
    inj_chk   = next((c for c in all_chks if hits(c["claim"], inj_s, inj_o)), None)
    verdict   = (f"{inj_chk['verdict']} / {inj_chk['confidence']} / {inj_chk.get('reason','')}"
                 if inj_chk else "NOT EXTRACTED by triple extractor")

    status = "✅ DETECTED" if detected else "❌ MISSED"
    print(f"  Result   : {status}")
    print(f"  Detector : {verdict}")
    if errors:
        print(f"  All flags: {[e['claim'] for e in errors]}")

    results.append({
        "img_id":       img_id,
        "pil_img":      pil_img,
        "base_cap":     base_cap,
        "inject_sent":  inject_sent,
        "hallucinated": hallucinated,
        "inj_s": inj_s, "inj_r": inj_r, "inj_o": inj_o,
        "source_q":     source_q,
        "detected":     detected,
        "verdict":      verdict,
        "all_chks":     all_chks,
        "errors":       errors,
        "corrected":    result.get("corrected", hallucinated),
    })

# ── Summary ───────────────────────────────────────────────────────────────
n      = len(results)
caught = sum(r["detected"] for r in results)
print(f"\n{'='*60}")
print(f"DETECTION SUMMARY  ({n} images tested)")
print(f"{'='*60}")
if n:
    print(f"  Detected : {caught}/{n}  (recall {100*caught/n:.0f}%)")
    missed = [r for r in results if not r["detected"]]
    if missed:
        print(f"\nMissed ({len(missed)}):")
        for r in missed:
            print(f"  [{r['img_id']}] {r['inj_s']} {r['inj_r']} {r['inj_o']}")
            print(f"    → {r['verdict']}")
else:
    print("  No images tested (check candidates list above)")


In [ ]:
# ============================================================
# Cell 4 — Visualize Detection Results
# ============================================================
# For each tested image shows:
#   Left panel  : image with GDino bounding boxes (if Cell 1b was run)
#                 Blue   = entity detected in caption
#                 Red    = entities from the INJECTED false claim
#   Right panel : caption diff + triple verdict table
#                 Injected text  → red highlight
#                 Corrected text → orange highlight (if changed)
#                 Unchanged      → green highlight (if correct)
# ============================================================

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import textwrap, difflib
from IPython.display import display, HTML

def _html_diff_caption(base, injected, corrected, inj_sent):
    """
    Three-row HTML table showing base / hallucinated / corrected captions.
    Injected sentence is highlighted red; corrected changes highlighted orange.
    """
    def hl(text, color, bold=False):
        b = "<b>" if bold else ""
        be = "</b>" if bold else ""
        return f'<span style="background:{color};border-radius:3px;padding:1px 3px">{b}{text}{be}</span>'

    # Row 1: base caption (clean)
    row1 = f"<b>Base caption:</b><br>{base}"

    # Row 2: hallucinated — highlight injected sentence in red
    inj_idx = injected.find(inj_sent.strip())
    if inj_idx >= 0:
        hi_row2 = (injected[:inj_idx]
                   + hl(injected[inj_idx:inj_idx + len(inj_sent.strip())], "#ffcccc", bold=True)
                   + injected[inj_idx + len(inj_sent.strip()):])
    else:
        hi_row2 = injected
    row2 = f"<b>Hallucinated (input to detector):</b><br>{hi_row2}"

    # Row 3: corrected — diff vs hallucinated, highlight changes in orange
    matcher = difflib.SequenceMatcher(None, injected.split(), corrected.split())
    parts = []
    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        chunk = " ".join(corrected.split()[j1:j2])
        if tag == "equal":
            parts.append(chunk)
        elif tag in ("replace", "insert"):
            parts.append(hl(chunk, "#ffe0b2", bold=True))   # orange
        # delete: just skip (text removed)
    hi_row3 = " ".join(parts)
    changed = corrected != injected
    status_badge = (hl("CORRECTED", "#c8e6c9") if changed else hl("UNCHANGED", "#e0e0e0"))
    row3 = f"<b>Corrected output {status_badge}:</b><br>{hi_row3}"

    return f"""
    <div style="font-family:monospace;font-size:13px;border:1px solid #ccc;
                border-radius:6px;padding:12px;margin:8px 0;background:#fafafa">
      <div style="margin-bottom:8px">{row1}</div>
      <hr style="border:0;border-top:1px solid #ddd">
      <div style="margin-bottom:8px">{row2}</div>
      <hr style="border:0;border-top:1px solid #ddd">
      <div>{row3}</div>
    </div>"""

def _html_verdict_table(all_chks, inj_s, inj_o):
    """HTML table of all triple verdicts; injected triple is highlighted."""
    rows = ""
    for c in all_chks:
        is_injected = (inj_s.lower() in c["claim"].lower() and
                       inj_o.lower() in c["claim"].lower())
        v = c.get("verdict", "?")
        conf = c.get("confidence", "?")
        reason = c.get("reason", "")[:80]
        verdict_color = {"CORRECT": "#c8e6c9", "INCORRECT": "#ffcdd2",
                         "UNKNOWN": "#fff9c4"}.get(v, "#e0e0e0")
        row_bg = "#fff3e0" if is_injected else "white"
        tag = " ← <b>INJECTED</b>" if is_injected else ""
        rows += (f"<tr style='background:{row_bg}'>"
                 f"<td style='padding:4px 8px'>{c['claim']}{tag}</td>"
                 f"<td style='padding:4px 8px'><span style='background:{verdict_color};"
                 f"border-radius:3px;padding:1px 4px'>{v}</span></td>"
                 f"<td style='padding:4px 8px'>{conf}</td>"
                 f"<td style='padding:4px 8px;font-size:11px;color:#555'>{reason}</td></tr>")
    return f"""
    <table style="border-collapse:collapse;font-family:monospace;font-size:12px;width:100%">
      <tr style="background:#eeeeee">
        <th style="padding:4px 8px;text-align:left">Triple</th>
        <th style="padding:4px 8px;text-align:left">Verdict</th>
        <th style="padding:4px 8px;text-align:left">Conf</th>
        <th style="padding:4px 8px;text-align:left">Reason</th>
      </tr>{rows}
    </table>"""


def visualize_result(res):
    img_id    = res["img_id"]
    pil_img   = res["pil_img"]
    inj_s, inj_r, inj_o = res["inj_s"], res["inj_r"], res["inj_o"]
    detected  = res["detected"]
    status_str = "✅ DETECTED" if detected else "❌ MISSED"

    # ── Extract all entities from the hallucinated caption for GDino ──────
    import spacy as _spacy
    _doc = nlp(res["hallucinated"])
    cap_entities = list({chunk.text.lower() for chunk in _doc.noun_chunks
                         if len(chunk.text.split()) <= 3})

    # ── Run GDino (if loaded) ─────────────────────────────────────────────
    dets = _run_gdino(pil_img, cap_entities) if gdino_model is not None else []
    inj_entities = {inj_s.lower(), inj_o.lower()}

    # ── Figure: image + boxes ─────────────────────────────────────────────
    fig, ax = plt.subplots(1, 1, figsize=(8, 6))
    ax.imshow(pil_img)
    ax.set_title(f"{img_id}  |  {status_str}", fontsize=12, fontweight="bold",
                 color="green" if detected else "red")
    ax.axis("off")

    W, H = pil_img.size
    for d in dets:
        x1n, y1n, x2n, y2n = d["box"]
        x1, y1, x2, y2 = x1n*W, y1n*H, x2n*W, y2n*H
        label = d["label"]
        score = d["score"]
        # Red box if entity is part of injected false claim; else blue
        is_inj_entity = any(e in label.lower() or label.lower() in e
                             for e in inj_entities)
        color = "red" if is_inj_entity else "dodgerblue"
        lw    = 3    if is_inj_entity else 2
        rect  = mpatches.FancyBboxPatch(
            (x1, y1), x2-x1, y2-y1,
            boxstyle="round,pad=2", linewidth=lw,
            edgecolor=color, facecolor="none"
        )
        ax.add_patch(rect)
        ax.text(x1+2, max(y1-6, 0), f"{label} {score:.2f}",
                color="white", fontsize=8, fontweight="bold",
                bbox=dict(facecolor=color, alpha=0.8, pad=1, edgecolor="none"))

    # Legend
    legend_elements = [
        mpatches.Patch(facecolor="dodgerblue", label="Detected entity (caption)"),
        mpatches.Patch(facecolor="red",        label="Entity in injected false claim"),
    ]
    if dets:
        ax.legend(handles=legend_elements, loc="lower left", fontsize=8,
                  framealpha=0.85)
    elif gdino_model is None:
        ax.text(0.02, 0.02, "Run Cell 1b to see GDino detections",
                transform=ax.transAxes, fontsize=9, color="gray",
                verticalalignment="bottom")

    plt.tight_layout()
    plt.show()

    # ── HTML: caption diff + verdict table ───────────────────────────────
    display(HTML(f"<h4 style='margin:4px 0'>R-Bench source: <i>{res['source_q']}</i></h4>"))
    display(HTML(f"<b>Injected hallucination:</b> "
                 f"<span style='background:#ffcccc;padding:2px 6px;border-radius:3px'>"
                 f"{inj_s} <u>{inj_r}</u> {inj_o}</span>"))
    display(HTML(_html_diff_caption(
        res["base_cap"], res["hallucinated"], res["corrected"], res["inject_sent"]
    )))
    display(HTML("<b>Triple verdicts:</b>"))
    display(HTML(_html_verdict_table(res["all_chks"], inj_s, inj_o)))
    display(HTML("<hr>"))


# ── Run visualization for all results ─────────────────────────────────────
print(f"Visualizing {len(results)} results...\n")
for res in results:
    visualize_result(res)
